# Workit RAG 평가(eval) on Colab (GPU)

이 노트북은 로컬에서 만든 RAG 파이프라인(병렬 JoRAG/HoRAG + 계층형 Cascaded)을
Colab GPU에서 돌리기 위한 것입니다. 로컬 Qdrant 서버에 접속하는 대신, 이미
임베딩이 끝난 파일들을 Colab에 직접 업로드해서 **Colab 세션 안에서 로컬(파일
기반) Qdrant를 새로 만들고** 그 위에서 sweep을 실행합니다.

## 실행 전 체크리스트

1. **런타임 → 런타임 유형 변경 → GPU(T4)** 로 바꾸세요.
2. 왼쪽 파일 탐색기(📁)에서 `/content/` 에 아래 파일들을 업로드하세요.
   - `vectors_jo_fixedid.npz`
   - `vectors_ho_fixedid.npz`
   - `sparse_weights_jo_fixedid.json`
   - `sparse_weights_ho_fixedid.json`
   - `chunks_jo_fixedid.json`
   - `chunks_ho_fixedid.json`
   - `gold_standard_v3.json` (실버 스탠다드 평가셋 — 파일명이 다르면 아래 셀에서 경로만 바꿔주세요)
3. 위 파일들이 다 올라간 뒤 셀을 위에서부터 순서대로 실행하세요.

## 노트북 구성
1. 패키지 설치 / GPU 확인
2. 파이프라인 코드 작성 (`yoonha_law_rag.py`, `yoonha_law_rag_cascaded.py`, `yoonha_colab_upsert.py`)
3. 업로드 파일 존재 확인
4. 로컬 Qdrant 생성 + upsert
5. 병렬(JoRAG/HoRAG) sweep 실행
6. 계층형(Cascaded) sweep 실행
7. 두 결과 비교


## 1. 패키지 설치 / GPU 확인

In [1]:
!pip install qdrant-client FlagEmbedding transformers torch pandas --quiet


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 398.1/398.1 kB 15.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 247.7/247.7 kB 10.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 947.5/947.5 kB 28.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 34.0 MB/s eta 0:00:00


In [2]:
import torch
print("CUDA 사용 가능:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("⚠️  GPU가 안 잡혔습니다. 런타임 → 런타임 유형 변경 → GPU(T4)로 바꾼 뒤 다시 실행하세요.")


CUDA 사용 가능: True
GPU: Tesla T4


## 2. 파이프라인 코드 작성

아래 3개 셀은 `%%writefile`로 `/content/`에 실제 `.py` 파일을 만듭니다.
로컬에서 검증한 코드와 100% 동일한 내용입니다 (병렬 캐싱 구조 포함).

In [3]:
%%writefile /content/yoonha_law_rag.py
"""
Workit - 계약서 검토 RAG 파이프라인 (2-flow, sweep 가능한 버전 + 캐싱)
파일명: rag/yoonha_law_rag.py

전체 흐름 (다이어그램 기준):
  1) JoRAG   : law_kb_jo_fixedid 에서 "조" 단위로 직접 검색 (넓은 단위, 청킹 크기 큼)
  2) HoRAG   : law_kb_ho_fixedid 에서 "호/목/세목"까지 쪼갠 세부 단위로 검색
               → 후보들의 cross_refs로 관련 조항 추가 확장 (별도 xref 컬렉션 없음,
                 cross_refs가 이미 ho payload 안에 들어있음)
               → 항상 parent_chunk_id로 "조" 텍스트를 fetch해서 최종 출력 단위를
                 조로 통일 (하위 단위는 검색에만 쓰고 최종 결과로는 노출 안 함)
  두 flow 모두 계약서 조항 1개당 결과를 병합해서 list[ClauseResult]로 반환.

sweep 가능하게 바뀐 점 (이전 버전과 차이):
  - alpha, reranker1/2 사용 여부, rerank1_k, rerank2_k, fetch_k, top_k를
    전부 모듈 상수가 아니라 함수 인자로 뺐다. 최적 조합(alpha, reranking on/off)은
    아직 확정 전이라 grid search로 찾아야 하고, 그러려면 이 값들이 호출부에서
    자유롭게 바뀔 수 있어야 한다.
  - 모듈 상수는 "sweep 안 할 때 쓰는 기본값" 역할만 한다 (DEFAULT_* 접두사).
  - use_reranker1 / use_reranker2 플래그를 명시적으로 추가했다. reranker 객체
    자체는 로드 비용이 커서 미리 한 번만 로드해두고, 이 플래그로 "이번 호출에서
    쓸지 말지"만 토글하는 방식 (매번 로드하지 않도록).

캐싱 구조 (이번에 추가된 부분):
  - alpha는 RRF 가중치 합산에만 쓰이고, 임베딩/Qdrant raw 검색 결과와는 무관하다.
    그래서 alpha를 sweep할 때 (collection, query_text)별 raw 검색 결과를 한 번만
    구하고, alpha 조합마다는 캐시된 raw 결과로 RRF 점수만 다시 계산한다.
  - reranker 점수는 (reranker_name, query_text, chunk_id) 쌍에만 의존한다.
    alpha가 바뀌면 어떤 chunk_id가 rerank1_k/rerank2_k 안에 들어오는지는 달라질
    수 있지만, 같은 chunk_id에 대한 점수 자체는 항상 같다. 그래서 미스(cache miss)만
    골라서 계산하고 나머지는 캐시에서 꺼내 쓴다 (memoization).
  - parent fetch(조 텍스트)와 cross-ref 확장(scroll 조회)도 chunk_id -> payload
    조회라 alpha/쿼리와 무관하게 전역 캐시가 가능하다. 여러 조항이 같은 법 조항을
    참조하는 경우가 많아서 쿼리 간에도 재사용된다.
  - cache=None으로 호출하면 캐싱 없이 기존과 완전히 동일하게 동작한다. sweep이
    아닌 일반 호출(review_contract_jo/ho 등 단발 검색)에서는 굳이 캐시를 넘길
    필요 없다.

컬렉션:
  - JoRAG : law_kb_jo_fixedid (조 단위, parent 없음)
  - HoRAG : law_kb_ho_fixedid (호/목/세목 단위 + cross_refs, parent fetch → 조 텍스트)

공통 출력: list[ClauseResult]  ← 항상 조 단위로 반환
"""

from __future__ import annotations

import json
import re
from dataclasses import dataclass, field, asdict
from pathlib import Path

import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from FlagEmbedding import BGEM3FlagModel
from qdrant_client import QdrantClient
from qdrant_client.models import SparseVector, Filter, FieldCondition, MatchValue

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 경로 / 설정
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
_THIS_DIR     = Path(__file__).resolve().parent
_DATA_DIR     = _THIS_DIR.parent / "data"
LAWS_REF_PATH = _DATA_DIR / "hn_seed" / "law_refs.json"

QDRANT_HOST = "localhost"
QDRANT_PORT = 6333

COLLECTION_JO = "law_kb_jo_fixedid"
COLLECTION_HO = "law_kb_ho_fixedid"
# HoXrefRAG는 별도 컬렉션 없이 COLLECTION_HO를 그대로 씀 (cross_refs가 payload에 이미 있음)

EMBED_MODEL = "BAAI/bge-m3"

# sweep 안 할 때 쓰는 기본값. 실제 최적값은 yoonha_rag_eval.py로 찾는다.
DEFAULT_FETCH_K    = 50
DEFAULT_RERANK1_K  = 30
DEFAULT_RERANK2_K  = 10
DEFAULT_TOP_K      = 10
DEFAULT_ALPHA      = 0.5   # 1.0 = dense only, 0.0 = sparse only

RERANKER1_MODEL = "Dongjin-kr/ko-reranker"
RERANKER2_MODEL = "BAAI/bge-reranker-v2-m3"


# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# Sweep 캐시
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

@dataclass
class SweepCache:
    """
    alpha × reranker on/off sweep에서 재사용 가능한 중간 계산 결과를 담는 캐시.

    - embed        : query_text -> (dense_vec, sparse_vec)
                      (임베딩은 collection/alpha와 무관 — jo/ho variant 간에도 공유됨)
    - raw_search   : (collection, query_text, fetch_k) -> (dense_points, sparse_points)
                      (alpha와 무관한 Qdrant 원본 검색 결과. RRF 합산만 alpha마다 다시 함)
    - scroll       : (collection, chunk_id) -> payload
                      (parent fetch / cross-ref 확장에서 쓰는 단건 조회. 여러 쿼리에서
                       같은 법 조항을 참조하면 자동으로 재사용됨)
    - rerank_score : (reranker_name, query_text, chunk_id) -> score
                      (reranker1/2 cross-encoder 점수. alpha가 달라져도 같은 chunk_id에
                       대한 점수는 항상 같으므로 미스만 계산)

    cache=None으로 함수를 호출하면 캐싱 없이 기존과 동일하게 동작한다.
    """
    embed        : dict = field(default_factory=dict)
    raw_search   : dict = field(default_factory=dict)
    scroll       : dict = field(default_factory=dict)
    rerank_score : dict = field(default_factory=dict)

    def stats(self) -> dict:
        return {
            "embed_cached":      len(self.embed),
            "raw_search_cached": len(self.raw_search),
            "scroll_cached":     len(self.scroll),
            "rerank_cached":     len(self.rerank_score),
        }


# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# Cross-encoder Reranker
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

class CrossEncoderReranker:
    """
    transformers AutoModel 기반 Cross-encoder reranker.
    FlagReranker 대체용 — 최신 transformers 호환.
    """

    def __init__(self, model_name: str, device: str = "cpu"):
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model     = AutoModelForSequenceClassification.from_pretrained(model_name)
        self.model.to(device)
        self.model.eval()
        self.device = device

    def compute_score(
        self,
        pairs     : list[list[str]],
        batch_size: int  = 32,
        normalize : bool = True,
    ) -> list[float]:
        all_scores: list[float] = []

        for i in range(0, len(pairs), batch_size):
            batch   = pairs[i : i + batch_size]
            encoded = self.tokenizer(
                [p[0] for p in batch],
                [p[1] for p in batch],
                padding=True,
                truncation=True,
                max_length=512,
                return_tensors="pt",
            )
            encoded = {k: v.to(self.device) for k, v in encoded.items()}

            with torch.no_grad():
                logits = self.model(**encoded).logits

            scores = logits.squeeze(-1) if logits.shape[-1] == 1 else logits[:, 1]
            if normalize:
                scores = torch.sigmoid(scores)

            all_scores.extend(scores.cpu().tolist())

        return all_scores


# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 데이터 클래스
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

@dataclass
class LawRef:
    """검색된 법령 조문 1건."""
    chunk_id   : str
    article    : str
    category   : str
    law_name   : str
    chunk_text : str
    score      : float
    is_risk_ref: bool
    parent_id  : str = ""
    cross_refs : list[str] = field(default_factory=list)  # HoRAG(xref 확장) 전용


@dataclass
class ClauseResult:
    """계약서 조항 1건의 검색 결과 — 항상 조 단위."""
    clause_number: str
    clause_text  : str
    page         : int            = 0
    bbox         : dict | None    = None
    law_refs     : list[LawRef]   = field(default_factory=list)
    categories   : list[str]      = field(default_factory=list)


# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 공통 유틸
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

def load_laws_ref(path: Path = LAWS_REF_PATH) -> dict[str, dict]:
    if not path.exists():
        print(f"  ⚠️  laws_ref.json 없음: {path}")
        return {}
    with open(path, encoding="utf-8") as f:
        return json.load(f)


def load_embed_model(model_name: str = EMBED_MODEL, use_fp16: bool = True) -> BGEM3FlagModel:
    print(f"📦 임베딩 모델 로드: {model_name}")
    return BGEM3FlagModel(model_name, use_fp16=use_fp16)


def load_rerankers(device: str = "cpu") -> tuple[CrossEncoderReranker, CrossEncoderReranker]:
    """
    reranker는 로드 비용이 커서 sweep 중에는 한 번만 로드해두고,
    실제로 쓸지 말지는 각 검색 함수의 use_reranker1/use_reranker2 플래그로 토글한다.
    """
    print(f"📦 Re-ranker 1단계 로드: {RERANKER1_MODEL}")
    r1 = CrossEncoderReranker(RERANKER1_MODEL, device=device)
    print(f"📦 Re-ranker 2단계 로드: {RERANKER2_MODEL}")
    r2 = CrossEncoderReranker(RERANKER2_MODEL, device=device)
    return r1, r2


def derive_jo_id(chunk_id: str) -> str:
    """
    ho-level chunk_id 문자열에서 그 조에 해당하는 jo-level chunk_id를 직접 역산한다.

    ho id 형식: {prefix}_{장}_{절}_{조}_{항}_{호}_{목}_{세목} (8토큰 고정)
    jo id 형식: 일반 법령은 {prefix}_{장}_{절}_{조} (앞 4토큰),
                PYG(예규)는 조가 없어 항이 anchor이므로 {prefix}_{장}_{절}_{조=0}_{항} (앞 5토큰).

    주의: payload의 parent_chunk_id 필드는 이 용도로 쓰면 안 된다. 실제 데이터를
    검증해보면 parent_chunk_id는 JO 컬렉션이 아니라 HO 컬렉션 자기 자신 안의 다른
    chunk(조 단위로 롤업된 chunk, 혹은 중간 단계인 호/목)를 가리키고 있어서 JO
    컬렉션 chunk_id와 절대 일치하지 않는다 (검증: ho 7640개 중 parent_chunk_id가
    JO chunk_id와 일치한 건 0개). 반면 이 함수처럼 chunk_id 자신의 앞쪽 토큰만
    잘라내는 방식은 ho 7640개 전부 100% 올바른 jo_id로 매핑된다 (leaf 청크든
    조 단위 롤업 청크든 동일하게 성립).
    """
    tokens = chunk_id.split("_")
    if tokens[0] == "PYG":
        return "_".join(tokens[:5])
    return "_".join(tokens[:4])


def get_vectors(
    text : str,
    model: BGEM3FlagModel,
    cache: SweepCache | None = None,
) -> tuple[list[float], dict[int, float]]:
    """
    BGE-M3 dense/sparse 임베딩. collection이나 alpha와 무관하므로
    query_text만으로 캐시 가능 (jo/ho variant 간에도 재사용됨).
    """
    if cache is not None and text in cache.embed:
        return cache.embed[text]

    output = model.encode(
        [text],
        return_dense=True,
        return_sparse=True,
        return_colbert_vecs=False,
    )
    dense_vec       = output["dense_vecs"][0].tolist()
    lexical_weights = output["lexical_weights"][0]

    sparse_vec: dict[int, float] = {}
    for token_str, weight in lexical_weights.items():
        token_id = model.tokenizer.convert_tokens_to_ids(token_str)
        if isinstance(token_id, int):
            sparse_vec[token_id] = sparse_vec.get(token_id, 0.0) + float(weight)

    if cache is not None:
        cache.embed[text] = (dense_vec, sparse_vec)

    return dense_vec, sparse_vec


# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 계약서 청킹 (조 단위 출력)
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

def chunk_contract(text: str) -> list[dict]:
    """계약서를 조 단위로 청킹."""
    HANG_MAP = {c: i + 1 for i, c in enumerate("①②③④⑤⑥⑦⑧⑨⑩⑪⑫⑬⑭⑮")}
    HO_SPLIT_PATTERN = r"(?:^|\s)(\d{1,2}\.\s)"

    text = text.strip()
    header_pattern = re.compile(r"제(\d+)조(?:의(\d+))?\s*\(([^)]*)\)")
    raw_matches    = list(header_pattern.finditer(text))

    candidates = []
    for m in raw_matches:
        prefix = text[max(0, m.start() - 5):m.start()]
        if re.search(r"법\s*$", prefix):
            continue
        num           = int(m.group(1))
        sub           = m.group(2)
        clause_number = f"제{m.group(1)}조" + (f"의{sub}" if sub else "")
        candidates.append((num, clause_number, m.start()))

    header_spans = []
    last_num = 0
    for num, clause_number, start in candidates:
        if num >= last_num and num <= last_num + 5:
            header_spans.append((clause_number, start))
            last_num = num

    def split_into_ho(parent_number: str, unit_text: str) -> list[dict]:
        ho_splits = re.split(HO_SPLIT_PATTERN, unit_text)
        if len(ho_splits) <= 1:
            return [{"clause_number": parent_number, "clause_text": unit_text}]

        head   = ho_splits[0].strip()
        chunks = []
        if head:
            chunks.append({"clause_number": parent_number, "clause_text": head})

        k, last_ho_num = 1, 0
        while k < len(ho_splits) - 1:
            marker       = ho_splits[k].strip()
            ho_num_match = re.match(r"(\d{1,2})\.", marker)
            ho_num       = int(ho_num_match.group(1)) if ho_num_match else (k // 2 + 1)
            ho_body      = ho_splits[k + 1].strip() if k + 1 < len(ho_splits) else ""

            if ho_num == last_ho_num + 1 and ho_body:
                chunks.append({
                    "clause_number": f"{parent_number}제{ho_num}호",
                    "clause_text":   re.sub(r"\s+", " ", f"{marker} {ho_body}").strip(),
                })
                last_ho_num = ho_num
            elif ho_body:
                if chunks:
                    chunks[-1]["clause_text"] += f" {marker} {ho_body}"
                else:
                    chunks.append({"clause_number": parent_number, "clause_text": f"{marker} {ho_body}"})
            k += 2

        return chunks if chunks else [{"clause_number": parent_number, "clause_text": unit_text}]

    clauses = []
    for idx, (clause_number, start) in enumerate(header_spans):
        end       = header_spans[idx + 1][1] if idx + 1 < len(header_spans) else len(text)
        raw_block = text[start:end].strip()

        m          = header_pattern.match(raw_block)
        raw_header = m.group(0) if m else clause_number
        body       = raw_block[m.end():].strip() if m else raw_block

        if not body:
            continue

        hang_splits = re.split(r"([①②③④⑤⑥⑦⑧⑨⑩⑪⑫⑬⑭⑮])", body)

        if len(hang_splits) <= 1:
            clause_text = re.sub(r"\s+", " ", f"{raw_header} {body}").strip()
            clauses.extend(split_into_ho(clause_number, clause_text))
        else:
            j = 1
            while j < len(hang_splits) - 1:
                hang_char   = hang_splits[j]
                hang_body   = hang_splits[j + 1].strip() if j + 1 < len(hang_splits) else ""
                hang_num    = HANG_MAP.get(hang_char, j)
                if hang_body:
                    hang_text = re.sub(r"\s+", " ", f"{raw_header} {hang_char}{hang_body}").strip()
                    clauses.extend(split_into_ho(f"{clause_number}제{hang_num}항", hang_text))
                j += 2

    if not clauses:
        paragraphs = [p.strip() for p in re.split(r"\n\s*\n", text) if p.strip()]
        clauses = [
            {"clause_number": f"단락{i + 1}", "clause_text": para}
            for i, para in enumerate(paragraphs)
        ]

    return clauses


# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 공통 검색 / 리랭크 / parent fetch
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

def _hybrid_search(
    clause_text: str,
    client     : QdrantClient,
    model      : BGEM3FlagModel,
    collection : str,
    fetch_k    : int,
    alpha      : float,
    cache      : SweepCache | None = None,
) -> list[dict]:
    """
    Dense + Sparse 하이브리드 검색 (수동 RRF). alpha=1.0이면 dense만, 0.0이면 sparse만.

    alpha는 아래 RRF 합산 단계에서만 쓰이므로, Qdrant raw 검색 결과
    (dense_results, sparse_results)는 (collection, clause_text, fetch_k)가
    같으면 alpha와 무관하게 재사용 가능 — cache가 있으면 캐시에서 꺼내온다.
    """
    cache_key = (collection, clause_text, fetch_k)

    if cache is not None and cache_key in cache.raw_search:
        dense_results, sparse_results = cache.raw_search[cache_key]
    else:
        dense_vec, sparse_vec = get_vectors(clause_text, model, cache)
        indices = list(sparse_vec.keys())
        values  = list(sparse_vec.values())

        try:
            dense_results = client.query_points(
                collection_name=collection,
                query=dense_vec,
                using="dense",
                limit=fetch_k,
                with_payload=True,
            ).points

            sparse_results = client.query_points(
                collection_name=collection,
                query=SparseVector(indices=indices, values=values),
                using="sparse",
                limit=fetch_k,
                with_payload=True,
            ).points

        except Exception as e:
            print(f"  ⚠️  sparse 검색 실패, dense만 사용: {e}")
            dense_results = client.query_points(
                collection_name=collection,
                query=dense_vec,
                using="dense",
                limit=fetch_k,
                with_payload=True,
            ).points
            sparse_results = []

        if cache is not None:
            cache.raw_search[cache_key] = (dense_results, sparse_results)

    RRF_K = 60
    scores: dict[str, dict] = {}

    for rank, point in enumerate(dense_results, 1):
        cid = point.payload.get("chunk_id", str(point.id))
        scores[cid] = {
            "payload":     point.payload,
            "dense_rank":  rank,
            "sparse_rank": len(dense_results) + 1,
        }

    for rank, point in enumerate(sparse_results, 1):
        cid = point.payload.get("chunk_id", str(point.id))
        if cid in scores:
            scores[cid]["sparse_rank"] = rank
        else:
            scores[cid] = {
                "payload":     point.payload,
                "dense_rank":  len(sparse_results) + 1,
                "sparse_rank": rank,
            }

    results = []
    for cid, info in scores.items():
        rrf_score = (
            alpha         * (1 / (RRF_K + info["dense_rank"]))
            + (1 - alpha) * (1 / (RRF_K + info["sparse_rank"]))
        )
        results.append({
            "chunk_id" : cid,
            "payload"  : info["payload"],
            "rrf_score": rrf_score,
        })

    results.sort(key=lambda x: x["rrf_score"], reverse=True)
    return results


def _rerank(
    query        : str,
    candidates   : list[dict],
    reranker     : CrossEncoderReranker,
    top_k        : int,
    reranker_name: str = "reranker",
    cache        : SweepCache | None = None,
) -> list[dict]:
    """
    reranker_name은 캐시 키 네임스페이스 구분용 (예: "jo_r1", "ho_r2").
    같은 (reranker_name, query, chunk_id) 조합은 alpha가 달라져도 점수가
    동일하므로, cache가 있으면 미스(cache miss)만 계산하고 나머지는 재사용한다.
    """
    if not candidates:
        return []

    if cache is None:
        texts  = [c["payload"].get("text", c["payload"].get("chunk_text", "")) for c in candidates]
        pairs  = [[query, t] for t in texts]
        scores = reranker.compute_score(pairs, normalize=True)
    else:
        scores: list[float | None] = [None] * len(candidates)
        miss_idx: list[int] = []

        for i, c in enumerate(candidates):
            key = (reranker_name, query, c["chunk_id"])
            if key in cache.rerank_score:
                scores[i] = cache.rerank_score[key]
            else:
                miss_idx.append(i)

        if miss_idx:
            miss_texts = [
                candidates[i]["payload"].get("text", candidates[i]["payload"].get("chunk_text", ""))
                for i in miss_idx
            ]
            miss_pairs  = [[query, t] for t in miss_texts]
            miss_scores = reranker.compute_score(miss_pairs, normalize=True)

            for i, s in zip(miss_idx, miss_scores):
                scores[i] = s
                cache.rerank_score[(reranker_name, query, candidates[i]["chunk_id"])] = s

    ranked = sorted(zip(scores, candidates), key=lambda x: x[0], reverse=True)
    return [item for _, item in ranked[:top_k]]


def _fetch_parent_texts(
    candidates: list[dict],
    client    : QdrantClient,
    parent_collection: str = COLLECTION_JO,
    cache     : SweepCache | None = None,
) -> list[dict]:
    """
    각 후보 ho chunk_id에서 derive_jo_id()로 조 단위 chunk_id를 역산해,
    그 조 텍스트를 law_kb_jo_fixedid에서 조회해 payload["text"]를 교체한다.

    payload의 parent_chunk_id 필드는 쓰지 않는다 (derive_jo_id 함수 docstring 참고
    — 그 필드는 JO 컬렉션이 아니라 HO 컬렉션 자기 자신을 가리키고 있어서 이 용도로
    쓰면 매번 조회가 실패한다). chunk_id 자신의 앞쪽 토큰만으로 역산하는 방식은
    leaf 청크든 조 단위 롤업 청크든 상관없이 항상 올바른 jo_id를 준다.
    """
    jo_ids = list({
        derive_jo_id(c["payload"].get("chunk_id", c["chunk_id"]))
        for c in candidates
    })

    if not jo_ids:
        return candidates

    parent_texts: dict[str, str] = {}
    to_fetch: list[str] = []

    for jid in jo_ids:
        cache_key = (parent_collection, jid)
        if cache is not None and cache_key in cache.scroll:
            payload = cache.scroll[cache_key]
            parent_texts[jid] = payload.get("text", payload.get("chunk_text", ""))
        else:
            to_fetch.append(jid)

    try:
        for jid in to_fetch:
            results = client.scroll(
                collection_name=parent_collection,
                scroll_filter=Filter(
                    must=[FieldCondition(key="chunk_id", match=MatchValue(value=jid))]
                ),
                limit=1,
                with_payload=True,
                with_vectors=False,
            )
            points = results[0]
            if points:
                p = points[0].payload
                parent_texts[jid] = p.get("text", p.get("chunk_text", ""))
                if cache is not None:
                    cache.scroll[(parent_collection, jid)] = p
    except Exception as e:
        print(f"  ⚠️  parent fetch 실패: {e}")
        return candidates

    updated = []
    for c in candidates:
        jid = derive_jo_id(c["payload"].get("chunk_id", c["chunk_id"]))
        if jid in parent_texts:
            updated_payload         = dict(c["payload"])
            updated_payload["text"] = parent_texts[jid]
            updated.append({**c, "payload": updated_payload})
        else:
            updated.append(c)

    return updated


def _build_law_refs(
    candidates : list[dict],
    laws_ref   : dict[str, dict],
    top_k      : int,
    with_xref  : bool = False,
) -> list[LawRef]:
    law_refs: list[LawRef] = []
    for c in candidates[:top_k]:
        payload  = c["payload"]
        chunk_id = payload.get("chunk_id", "")
        ref_meta = laws_ref.get(chunk_id, {})

        law_refs.append(LawRef(
            chunk_id    = chunk_id,
            article     = ref_meta.get("article",  payload.get("article_number", "")),
            category    = ref_meta.get("category", payload.get("category", "")),
            law_name    = payload.get("law_name",  ""),
            chunk_text  = payload.get("text", payload.get("chunk_text", "")),
            score       = round(float(c.get("rrf_score", 0.0)), 4),
            is_risk_ref = bool(payload.get("is_risk_ref", False)),
            parent_id   = payload.get("parent_chunk_id", "") or "",
            cross_refs  = payload.get("cross_refs", []) if with_xref else [],
        ))

    return law_refs


# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# RAG 1: JoRAG — 조 단위 검색
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

def _search_jo(
    clause_text  : str,
    client       : QdrantClient,
    model        : BGEM3FlagModel,
    laws_ref     : dict[str, dict],
    reranker1    : CrossEncoderReranker | None = None,
    reranker2    : CrossEncoderReranker | None = None,
    use_reranker1: bool  = False,
    use_reranker2: bool  = False,
    top_k        : int   = DEFAULT_TOP_K,
    alpha        : float = DEFAULT_ALPHA,
    fetch_k      : int   = DEFAULT_FETCH_K,
    rerank1_k    : int   = DEFAULT_RERANK1_K,
    rerank2_k    : int   = DEFAULT_RERANK2_K,
    cache        : SweepCache | None = None,
) -> list[LawRef]:
    """
    JoRAG: law_kb_jo_fixedid에서 조 단위로 직접 검색.
    parent fetch 없음 — 이미 조 단위가 최상위.
    """
    candidates = _hybrid_search(clause_text, client, model, COLLECTION_JO, fetch_k, alpha, cache)

    if use_reranker1 and reranker1 and candidates:
        candidates = _rerank(clause_text, candidates, reranker1, rerank1_k, "jo_r1", cache)
    if use_reranker2 and reranker2 and candidates:
        candidates = _rerank(clause_text, candidates, reranker2, rerank2_k, "jo_r2", cache)

    return _build_law_refs(candidates, laws_ref, top_k, with_xref=False)


def review_contract_jo(
    contract_text: str,
    client       : QdrantClient,
    model        : BGEM3FlagModel,
    laws_ref     : dict[str, dict] | None = None,
    reranker1    : CrossEncoderReranker | None = None,
    reranker2    : CrossEncoderReranker | None = None,
    use_reranker1: bool  = False,
    use_reranker2: bool  = False,
    top_k        : int   = DEFAULT_TOP_K,
    alpha        : float = DEFAULT_ALPHA,
    fetch_k      : int   = DEFAULT_FETCH_K,
    rerank1_k    : int   = DEFAULT_RERANK1_K,
    rerank2_k    : int   = DEFAULT_RERANK2_K,
    cache        : SweepCache | None = None,
) -> list[ClauseResult]:
    """JoRAG 메인 인터페이스."""
    if laws_ref is None:
        laws_ref = load_laws_ref()

    clauses = chunk_contract(contract_text)
    results : list[ClauseResult] = []
    print(f"[JoRAG] 총 {len(clauses)}개 청크 검색 중...")

    for i, clause in enumerate(clauses, 1):
        print(f"  [{i}/{len(clauses)}] {clause['clause_number']} ...", end="\r")

        law_refs   = _search_jo(
            clause["clause_text"], client, model, laws_ref,
            reranker1, reranker2, use_reranker1, use_reranker2,
            top_k, alpha, fetch_k, rerank1_k, rerank2_k, cache,
        )
        categories = list(dict.fromkeys(r.category for r in law_refs if r.category))

        results.append(ClauseResult(
            clause_number=clause["clause_number"],
            clause_text  =clause["clause_text"],
            law_refs     =law_refs,
            categories   =categories,
        ))

    print(f"\n[JoRAG] ✅ 완료")
    return results


# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# RAG 2: HoRAG — 호/목/세목 단위 검색 + cross_refs 확장 + parent fetch
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

def _expand_with_cross_refs(
    candidates: list[dict],
    client    : QdrantClient,
    cache     : SweepCache | None = None,
) -> list[dict]:
    """
    각 후보의 cross_refs(같은 law_kb_ho_fixedid payload 안의 필드)에 있는
    chunk_id를 추가 조회. 이미 후보에 있는 chunk_id는 중복 추가하지 않음.
    추가된 항목의 rrf_score는 0.0 (리랭크 단계에서 점수 재계산됨).

    chunk_id -> payload 조회라 쿼리/alpha와 무관 — 여러 조항이 같은 참조를
    가지면 캐시에서 재사용된다.
    """
    existing_ids  = {c["chunk_id"] for c in candidates}
    ref_ids_total : list[str] = []

    for c in candidates:
        cross_refs = c["payload"].get("cross_refs", [])
        for ref_id in cross_refs:
            if ref_id not in existing_ids and ref_id not in ref_ids_total:
                ref_ids_total.append(ref_id)

    if not ref_ids_total:
        return candidates

    payload_by_ref: dict[str, dict] = {}
    to_fetch: list[str] = []

    for ref_id in ref_ids_total:
        cache_key = (COLLECTION_HO, ref_id)
        if cache is not None and cache_key in cache.scroll:
            payload_by_ref[ref_id] = cache.scroll[cache_key]
        else:
            to_fetch.append(ref_id)

    try:
        for ref_id in to_fetch:
            results = client.scroll(
                collection_name=COLLECTION_HO,
                scroll_filter=Filter(
                    must=[FieldCondition(key="chunk_id", match=MatchValue(value=ref_id))]
                ),
                limit=1,
                with_payload=True,
                with_vectors=False,
            )
            points = results[0]
            if points:
                p = points[0].payload
                payload_by_ref[ref_id] = p
                if cache is not None:
                    cache.scroll[(COLLECTION_HO, ref_id)] = p
    except Exception as e:
        print(f"  ⚠️  cross_ref fetch 실패: {e}")

    extra_chunks = []
    for ref_id, p in payload_by_ref.items():
        extra_chunks.append({
            "chunk_id" : ref_id,
            "payload"  : p,
            "rrf_score": 0.0,   # 리랭크에서 점수 재계산됨
        })
        existing_ids.add(ref_id)

    return candidates + extra_chunks


def _search_ho(
    clause_text  : str,
    client       : QdrantClient,
    model        : BGEM3FlagModel,
    laws_ref     : dict[str, dict],
    reranker1    : CrossEncoderReranker | None = None,
    reranker2    : CrossEncoderReranker | None = None,
    use_reranker1: bool  = False,
    use_reranker2: bool  = False,
    use_cross_refs: bool = True,
    top_k        : int   = DEFAULT_TOP_K,
    alpha        : float = DEFAULT_ALPHA,
    fetch_k      : int   = DEFAULT_FETCH_K,
    rerank1_k    : int   = DEFAULT_RERANK1_K,
    rerank2_k    : int   = DEFAULT_RERANK2_K,
    cache        : SweepCache | None = None,
) -> list[LawRef]:
    """
    HoRAG: law_kb_ho_fixedid에서 호/목/세목 단위 검색
    → (옵션) cross_refs로 관련 조항 확장
    → parent_chunk_id로 law_kb_jo_fixedid에서 조 전체 텍스트 fetch
      (최종 출력 단위는 항상 조 — 하위 단위는 검색 후보로만 쓰고 결과로는 안 남김)
    """
    candidates = _hybrid_search(clause_text, client, model, COLLECTION_HO, fetch_k, alpha, cache)

    if use_reranker1 and reranker1 and candidates:
        candidates = _rerank(clause_text, candidates, reranker1, rerank1_k, "ho_r1", cache)

    if use_cross_refs:
        candidates = _expand_with_cross_refs(candidates, client, cache)

    # parent fetch: 호/목/세목 → 조 텍스트로 교체 (최종 출력 단위 통일)
    candidates = _fetch_parent_texts(candidates, client, parent_collection=COLLECTION_JO, cache=cache)

    if use_reranker2 and reranker2 and candidates:
        candidates = _rerank(clause_text, candidates, reranker2, rerank2_k, "ho_r2", cache)

    return _build_law_refs(candidates, laws_ref, top_k, with_xref=use_cross_refs)


def review_contract_ho(
    contract_text: str,
    client       : QdrantClient,
    model        : BGEM3FlagModel,
    laws_ref     : dict[str, dict] | None = None,
    reranker1    : CrossEncoderReranker | None = None,
    reranker2    : CrossEncoderReranker | None = None,
    use_reranker1: bool  = False,
    use_reranker2: bool  = False,
    use_cross_refs: bool = True,
    top_k        : int   = DEFAULT_TOP_K,
    alpha        : float = DEFAULT_ALPHA,
    fetch_k      : int   = DEFAULT_FETCH_K,
    rerank1_k    : int   = DEFAULT_RERANK1_K,
    rerank2_k    : int   = DEFAULT_RERANK2_K,
    cache        : SweepCache | None = None,
) -> list[ClauseResult]:
    """HoRAG 메인 인터페이스 (cross_refs 확장 포함, 별도 xref variant 없음)."""
    if laws_ref is None:
        laws_ref = load_laws_ref()

    clauses = chunk_contract(contract_text)
    results : list[ClauseResult] = []
    print(f"[HoRAG] 총 {len(clauses)}개 청크 검색 중...")

    for i, clause in enumerate(clauses, 1):
        print(f"  [{i}/{len(clauses)}] {clause['clause_number']} ...", end="\r")

        law_refs   = _search_ho(
            clause["clause_text"], client, model, laws_ref,
            reranker1, reranker2, use_reranker1, use_reranker2, use_cross_refs,
            top_k, alpha, fetch_k, rerank1_k, rerank2_k, cache,
        )
        categories = list(dict.fromkeys(r.category for r in law_refs if r.category))

        results.append(ClauseResult(
            clause_number=clause["clause_number"],
            clause_text  =clause["clause_text"],
            law_refs     =law_refs,
            categories   =categories,
        ))

    print(f"\n[HoRAG] ✅ 완료")
    return results


# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# JSON 변환
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

def results_to_json(results: list[ClauseResult]) -> list[dict]:
    return [asdict(r) for r in results]


# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 편의: 단일 조항 검색 (sweep 스크립트에서 이 함수들을 직접 호출)
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

def search_jo(clause_text: str, client: QdrantClient, model: BGEM3FlagModel,
              laws_ref: dict, reranker1=None, reranker2=None,
              use_reranker1=False, use_reranker2=False,
              top_k=DEFAULT_TOP_K, alpha=DEFAULT_ALPHA, fetch_k=DEFAULT_FETCH_K,
              rerank1_k=DEFAULT_RERANK1_K, rerank2_k=DEFAULT_RERANK2_K,
              cache: SweepCache | None = None) -> list[LawRef]:
    return _search_jo(clause_text, client, model, laws_ref, reranker1, reranker2,
                       use_reranker1, use_reranker2, top_k, alpha, fetch_k, rerank1_k, rerank2_k,
                       cache)


def search_ho(clause_text: str, client: QdrantClient, model: BGEM3FlagModel,
              laws_ref: dict, reranker1=None, reranker2=None,
              use_reranker1=False, use_reranker2=False, use_cross_refs=True,
              top_k=DEFAULT_TOP_K, alpha=DEFAULT_ALPHA, fetch_k=DEFAULT_FETCH_K,
              rerank1_k=DEFAULT_RERANK1_K, rerank2_k=DEFAULT_RERANK2_K,
              cache: SweepCache | None = None) -> list[LawRef]:
    return _search_ho(clause_text, client, model, laws_ref, reranker1, reranker2,
                       use_reranker1, use_reranker2, use_cross_refs,
                       top_k, alpha, fetch_k, rerank1_k, rerank2_k, cache)


Writing /content/yoonha_law_rag.py


In [4]:
%%writefile /content/yoonha_law_rag_cascaded.py
"""
Workit - 계층형(Cascaded) RAG 파이프라인
파일명: rag/yoonha_law_rag_cascaded.py

병렬 버전(yoonha_law_rag.py의 JoRAG/HoRAG 독립 검색)과는 완전히 별개로
관리되는 계층형(coarse-to-fine) variant.

흐름:
  1단계 (coarse) : law_kb_jo_fixedid에서 조 단위로 넓게 검색해 상위
                    jo_top_k개 조만 후보로 추린다 (JoRAG와 동일한 하이브리드
                    검색 로직 재사용 — _hybrid_search를 그대로 import).
  2단계 (fine)   : law_kb_ho_fixedid에서 호/목/세목 단위로 검색한 뒤,
                    각 후보의 chunk_id를 derive_jo_id()로 조 단위 id로 역산해서
                    1단계 후보 조 안에 있는 것만 Python 사이드에서 남긴다
                    (payload의 parent_chunk_id 필드는 HO 컬렉션 자기 자신을
                    가리키는 값이라 이 용도로 쓸 수 없음 — derive_jo_id 참고).
  3단계          : (옵션) cross_refs 확장 — 명시적 인용은 1단계 필터 밖이어도
                    살려서 확장한다 (조 단위 후보에 안 걸렸다고 인용 관계까지
                    끊어버리면 너무 공격적인 필터링이라고 판단).
  4단계          : parent fetch로 최종 출력을 조 텍스트로 통일 (병렬 버전과 동일).

주의:
  - 1단계 recall이 2단계 recall의 상한선이다. 1단계에서 정답 조가 top-k
    밖으로 밀리면 2단계에서 아무리 잘 검색해도 복구 불가능하다. 그래서
    jo_top_k는 반드시 sweep 대상에 넣어야 하고, 이 모듈은 "1단계에서
    정답 조가 살아남았는가"를 별도로 진단할 수 있는 get_stage1_jo_candidates도
    노출한다 (yoonha_rag_eval_cascaded.py에서 stage1_recall 지표로 사용).
  - PYG(예규)는 조가 없어 jo-level이 항 단위로 anchor된다. derive_jo_id()가
    PYG는 앞 5토큰({prefix}_{장}_{절}_{조=0}_{항})을, 일반 법령은 앞 4토큰을
    jo_id로 역산하도록 이미 분기 처리돼 있어서 필터링 자체는 정상 동작하지만,
    "항 단위로 넓게 검색 → 그 항에 속하는 호/목/세목만 좁혀서 검색"이라
    병렬 버전의 "조 단위 필터"와는 의미가 다르므로 평가 시 PYG만 따로
    recall을 봐두는 걸 권장한다.
  - 이 모듈은 yoonha_law_rag.py의 함수 이름을 재사용하지 않는다. 공용 유틸
    (get_vectors, CrossEncoderReranker, SweepCache, LawRef, ClauseResult,
    chunk_contract, _hybrid_search, _rerank, _fetch_parent_texts,
    _expand_with_cross_refs, _build_law_refs)은 그대로 import해서 쓰고,
    새로 정의하는 함수는 전부 *_cascaded 접미사를 붙였다.
"""

from __future__ import annotations

from qdrant_client import QdrantClient
from FlagEmbedding import BGEM3FlagModel

from yoonha_law_rag import (
    COLLECTION_JO,
    COLLECTION_HO,
    DEFAULT_FETCH_K,
    DEFAULT_RERANK1_K,
    DEFAULT_RERANK2_K,
    DEFAULT_TOP_K,
    DEFAULT_ALPHA,
    QDRANT_HOST,
    QDRANT_PORT,
    CrossEncoderReranker,
    LawRef,
    ClauseResult,
    SweepCache,
    chunk_contract,
    get_vectors,
    derive_jo_id,
    load_laws_ref,
    load_embed_model,
    load_rerankers,
    _hybrid_search,
    _rerank,
    _fetch_parent_texts,
    _expand_with_cross_refs,
    _build_law_refs,
)

# 계층형 전용 기본값 — 1단계(조 단위)에서 몇 개 조를 후보로 남길지.
# 너무 작으면 1단계에서 정답 조가 걸러져서 2단계가 복구 불가능해지고,
# 너무 크면 사실상 병렬 버전과 차이가 없어진다. sweep으로 찾아야 한다.
DEFAULT_JO_TOP_K = 20


# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 1단계: 조 단위 후보 추출
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

def get_stage1_jo_candidates(
    clause_text: str,
    client     : QdrantClient,
    model      : BGEM3FlagModel,
    alpha      : float = DEFAULT_ALPHA,
    fetch_k    : int   = DEFAULT_FETCH_K,
    jo_top_k   : int   = DEFAULT_JO_TOP_K,
    cache      : SweepCache | None = None,
) -> list[str]:
    """
    1단계: law_kb_jo_fixedid에서 조 단위로 넓게 검색해 상위 jo_top_k개
    조의 chunk_id만 반환한다. _hybrid_search를 그대로 재사용하므로
    JoRAG(병렬 버전)의 캐시(raw_search)와도 공유된다.

    sweep 스크립트에서 진단용으로 따로 호출해도, 이미 search_cascaded 내부에서
    같은 캐시 키로 호출된 적이 있으면 cache hit이라 추가 비용이 거의 없다.
    """
    candidates = _hybrid_search(clause_text, client, model, COLLECTION_JO, fetch_k, alpha, cache)
    return [c["chunk_id"] for c in candidates[:jo_top_k]]


# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 2단계: jo_id로 제한된 ho 후보 필터링 (Python 사이드)
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

def _hybrid_search_ho_filtered(
    clause_text       : str,
    client            : QdrantClient,
    model             : BGEM3FlagModel,
    fetch_k           : int,
    alpha             : float,
    allowed_parent_ids: list[str],
    cache             : SweepCache | None = None,
) -> list[dict]:
    """
    2단계: law_kb_ho_fixedid에서 검색한 뒤, 각 후보의 chunk_id를 derive_jo_id()로
    조 단위 id로 역산해서 allowed_parent_ids(1단계 후보 조) 안에 있는 것만 남긴다.

    처음에는 Qdrant Filter(parent_chunk_id MatchAny)로 서버 사이드 필터링을
    시도했었는데, 실제 데이터를 보면 payload의 parent_chunk_id 필드가 JO
    컬렉션이 아니라 HO 컬렉션 자기 자신(조 단위로 롤업된 다른 ho chunk 또는
    중간 단계인 호/목)을 가리키고 있어서 JO id와 절대 일치하지 않았다
    (검증 결과 0% 매치 — 그래서 이전 버전은 항상 빈 결과만 반환했음).
    derive_jo_id()가 chunk_id 문자열 자체에서 직접 역산하는 방식이라 이 필드에
    의존하지 않고, ho 7640개 전체에서 100% 정확하게 검증됐다.

    부가 효과: _hybrid_search를 필터 없이 그대로 재사용하므로, jo_top_k가
    달라져도 raw_search 캐시가 (collection, clause_text, fetch_k) 키 하나로
    공유된다 — 이전 버전(필터 지문을 캐시 키에 포함하던 방식)보다 캐시 재사용률이
    오히려 더 좋아졌다.
    """
    candidates = _hybrid_search(clause_text, client, model, COLLECTION_HO, fetch_k, alpha, cache)
    allowed_set = set(allowed_parent_ids)
    return [c for c in candidates if derive_jo_id(c["chunk_id"]) in allowed_set]


# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 전체 계층형 검색
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

def _search_cascaded(
    clause_text  : str,
    client       : QdrantClient,
    model        : BGEM3FlagModel,
    laws_ref     : dict[str, dict],
    reranker1    : CrossEncoderReranker | None = None,
    reranker2    : CrossEncoderReranker | None = None,
    use_reranker1: bool  = False,
    use_reranker2: bool  = False,
    use_cross_refs: bool = True,
    top_k        : int   = DEFAULT_TOP_K,
    alpha        : float = DEFAULT_ALPHA,
    fetch_k      : int   = DEFAULT_FETCH_K,
    rerank1_k    : int   = DEFAULT_RERANK1_K,
    rerank2_k    : int   = DEFAULT_RERANK2_K,
    jo_top_k     : int   = DEFAULT_JO_TOP_K,
    cache        : SweepCache | None = None,
) -> list[LawRef]:
    """
    1단계(JoRAG top-k) → 2단계(그 조들로 제한된 HoRAG) → cross-ref 확장
    → parent fetch(조 텍스트 통일) 순서로 진행하는 계층형 검색.
    """
    jo_candidates = _hybrid_search(clause_text, client, model, COLLECTION_JO, fetch_k, alpha, cache)
    allowed_parent_ids = [c["chunk_id"] for c in jo_candidates[:jo_top_k]]

    if not allowed_parent_ids:
        return []

    candidates = _hybrid_search_ho_filtered(
        clause_text, client, model, fetch_k, alpha, allowed_parent_ids, cache,
    )

    if use_reranker1 and reranker1 and candidates:
        candidates = _rerank(clause_text, candidates, reranker1, rerank1_k, "cascaded_r1", cache)

    if use_cross_refs:
        candidates = _expand_with_cross_refs(candidates, client, cache)

    # parent fetch: 호/목/세목 → 조 텍스트로 교체 (최종 출력 단위 통일, 병렬 버전과 동일)
    candidates = _fetch_parent_texts(candidates, client, parent_collection=COLLECTION_JO, cache=cache)

    if use_reranker2 and reranker2 and candidates:
        candidates = _rerank(clause_text, candidates, reranker2, rerank2_k, "cascaded_r2", cache)

    return _build_law_refs(candidates, laws_ref, top_k, with_xref=use_cross_refs)


def review_contract_cascaded(
    contract_text: str,
    client       : QdrantClient,
    model        : BGEM3FlagModel,
    laws_ref     : dict[str, dict] | None = None,
    reranker1    : CrossEncoderReranker | None = None,
    reranker2    : CrossEncoderReranker | None = None,
    use_reranker1: bool  = False,
    use_reranker2: bool  = False,
    use_cross_refs: bool = True,
    top_k        : int   = DEFAULT_TOP_K,
    alpha        : float = DEFAULT_ALPHA,
    fetch_k      : int   = DEFAULT_FETCH_K,
    rerank1_k    : int   = DEFAULT_RERANK1_K,
    rerank2_k    : int   = DEFAULT_RERANK2_K,
    jo_top_k     : int   = DEFAULT_JO_TOP_K,
    cache        : SweepCache | None = None,
) -> list[ClauseResult]:
    """계층형 RAG 메인 인터페이스 (review_contract_jo/ho와 동일한 사용 패턴)."""
    if laws_ref is None:
        laws_ref = load_laws_ref()

    clauses = chunk_contract(contract_text)
    results : list[ClauseResult] = []
    print(f"[Cascaded] 총 {len(clauses)}개 청크 검색 중...")

    for i, clause in enumerate(clauses, 1):
        print(f"  [{i}/{len(clauses)}] {clause['clause_number']} ...", end="\r")

        law_refs = _search_cascaded(
            clause["clause_text"], client, model, laws_ref,
            reranker1, reranker2, use_reranker1, use_reranker2, use_cross_refs,
            top_k, alpha, fetch_k, rerank1_k, rerank2_k, jo_top_k, cache,
        )
        categories = list(dict.fromkeys(r.category for r in law_refs if r.category))

        results.append(ClauseResult(
            clause_number=clause["clause_number"],
            clause_text  =clause["clause_text"],
            law_refs     =law_refs,
            categories   =categories,
        ))

    print(f"\n[Cascaded] ✅ 완료")
    return results


# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 편의: 단일 조항 검색 (sweep 스크립트에서 직접 호출)
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

def search_cascaded(clause_text: str, client: QdrantClient, model: BGEM3FlagModel,
                     laws_ref: dict, reranker1=None, reranker2=None,
                     use_reranker1=False, use_reranker2=False, use_cross_refs=True,
                     top_k=DEFAULT_TOP_K, alpha=DEFAULT_ALPHA, fetch_k=DEFAULT_FETCH_K,
                     rerank1_k=DEFAULT_RERANK1_K, rerank2_k=DEFAULT_RERANK2_K,
                     jo_top_k=DEFAULT_JO_TOP_K,
                     cache: SweepCache | None = None) -> list[LawRef]:
    return _search_cascaded(clause_text, client, model, laws_ref, reranker1, reranker2,
                             use_reranker1, use_reranker2, use_cross_refs,
                             top_k, alpha, fetch_k, rerank1_k, rerank2_k, jo_top_k, cache)


Writing /content/yoonha_law_rag_cascaded.py


In [5]:
%%writefile /content/yoonha_colab_upsert.py
"""
Workit - Colab용 로컬 Qdrant 셋업 + upsert 스크립트
파일명: rag/yoonha_colab_upsert.py

목적:
  Colab은 로컬 머신의 Qdrant(localhost:6333)에 접근할 수 없다. 대신 이미
  임베딩이 끝난 4개 파일(vectors_*.npz, sparse_weights_*.json)과 payload
  메타데이터(chunks_*_fixedid.json)를 Colab에 직접 업로드해서, Colab
  안에서 "로컬 파일 기반" Qdrant를 새로 띄우고 그 안에 upsert한다.
  서버 프로세스가 따로 필요 없다 (QdrantClient(path=...) 모드).

  이렇게 하면:
    - ngrok/터널링 없이 Colab 세션 안에서 완결됨
    - 로컬 머신을 계속 켜둘 필요 없음
    - GPU는 reranker/embedding 계산에만 쓰이고, Qdrant 자체는 CPU로 충분

전제:
  - vectors_jo_fixedid.npz / vectors_ho_fixedid.npz : {"vectors": (N,1024) float16,
    "chunk_ids": (N,) 문자열 배열}
  - sparse_weights_jo_fixedid.json / sparse_weights_ho_fixedid.json : 길이 N인
    리스트, vectors의 chunk_ids와 "같은 순서"로 정렬돼 있다고 이미 확인함
    (jo/ho 둘 다 순서 일치, 중복 0 — 실제 업로드 파일로 검증 완료).
  - chunks_jo_fixedid.json / chunks_ho_fixedid.json : chunk_id별 payload
    메타데이터(text, parent_chunk_id, cross_refs, law_name, article_number,
    title, hierarchy, is_ref_article, is_upper_law 등). vectors의 chunk_ids와
    "순서까지" 동일하다고 확인했지만, 이 스크립트는 순서에 의존하지 않고
    chunk_id로 다시 매핑해서 병합한다 (더 안전한 방식).

  주의 — 지금 chunks_*_fixedid.json에는 category, is_risk_ref 필드가 없다.
  yoonha_law_rag.py의 _build_law_refs가 이 필드들을 payload.get(key, 기본값)
  으로 읽기 때문에 없어도 에러는 안 나고 그냥 빈 문자열/False로 채워진다.
  나중에 laws_ref.json(load_laws_ref)에서 category를 따로 채워주는 구조라
  RAG 파이프라인 동작 자체엔 문제 없음.

사용법 (Colab 셀에서):
    !pip install qdrant-client --quiet
    !python yoonha_colab_upsert.py
  또는 노트북 안에서 build_local_qdrant() 함수를 직접 호출해도 된다.
"""

from __future__ import annotations

import json
from pathlib import Path

import numpy as np
from qdrant_client import QdrantClient
from qdrant_client.models import (
    Distance,
    VectorParams,
    SparseVectorParams,
    PointStruct,
    SparseVector,
)

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 경로 — Colab에서 업로드한 파일 위치에 맞게 수정
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
DATA_DIR = Path("/content")  # 업로드한 4+2개 파일이 있는 디렉터리

VECTORS_JO      = DATA_DIR / "vectors_jo_fixedid.npz"
VECTORS_HO      = DATA_DIR / "vectors_ho_fixedid.npz"
SPARSE_JO       = DATA_DIR / "sparse_weights_jo_fixedid.json"
SPARSE_HO       = DATA_DIR / "sparse_weights_ho_fixedid.json"
CHUNKS_JO       = DATA_DIR / "chunks_jo_fixedid.json"
CHUNKS_HO       = DATA_DIR / "chunks_ho_fixedid.json"

# Colab 세션 안에 로컬로 만들 Qdrant 저장 경로 (서버 프로세스 없이 파일 기반)
QDRANT_LOCAL_PATH = "/content/qdrant_local"

COLLECTION_JO = "law_kb_jo_fixedid"
COLLECTION_HO = "law_kb_ho_fixedid"

DENSE_DIM = 1024  # BGE-M3 dense 차원 (npz vectors.shape[1]로 실제 검증도 함께 함)


# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 로드 / 병합
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

def _load_merged(vectors_path: Path, sparse_path: Path, chunks_path: Path) -> list[dict]:
    """
    vectors(.npz) + sparse_weights(.json) + chunks(.json, payload)를
    chunk_id 기준으로 병합해서 [{chunk_id, dense_vec, sparse_vec, payload}, ...] 반환.
    """
    npz = np.load(vectors_path, allow_pickle=True)
    dense_vectors = npz["vectors"]          # (N, 1024) float16
    chunk_ids     = npz["chunk_ids"].tolist()

    assert dense_vectors.shape[1] == DENSE_DIM, (
        f"예상한 dense 차원({DENSE_DIM})과 실제({dense_vectors.shape[1]})가 다릅니다 — "
        f"임베딩 모델이 바뀐 건 아닌지 확인하세요."
    )

    with open(sparse_path, encoding="utf-8") as f:
        sparse_weights = json.load(f)  # chunk_ids와 같은 순서의 리스트

    assert len(sparse_weights) == len(chunk_ids), (
        f"sparse_weights 개수({len(sparse_weights)})와 chunk_ids 개수({len(chunk_ids)})가 다릅니다."
    )

    with open(chunks_path, encoding="utf-8") as f:
        chunks_list = json.load(f)

    payload_by_id = {c["chunk_id"]: c for c in chunks_list}

    missing = [cid for cid in chunk_ids if cid not in payload_by_id]
    if missing:
        print(f"  ⚠️  payload를 못 찾은 chunk_id {len(missing)}개 (예: {missing[:5]}) "
              f"— 해당 chunk는 text 없이 upsert됩니다.")

    merged = []
    for i, cid in enumerate(chunk_ids):
        payload = dict(payload_by_id.get(cid, {}))
        payload["chunk_id"] = cid  # payload 안에도 chunk_id를 넣어야 검색 결과에서 식별 가능
        merged.append({
            "chunk_id":   cid,
            "dense_vec":  dense_vectors[i].astype(np.float32).tolist(),
            "sparse_vec": sparse_weights[i],  # {"token_id": weight, ...}
            "payload":    payload,
        })

    return merged


def _to_qdrant_points(merged: list[dict]) -> list[PointStruct]:
    points = []
    for item in merged:
        sparse = item["sparse_vec"]
        indices = [int(k) for k in sparse.keys()]
        values  = [float(v) for v in sparse.values()]

        points.append(PointStruct(
            id=abs(hash(item["chunk_id"])) % (2**63),  # Qdrant point id는 int/UUID만 허용, chunk_id는 payload에 문자열로 그대로 보존
            vector={
                "dense":  item["dense_vec"],
                "sparse": SparseVector(indices=indices, values=values),
            },
            payload=item["payload"],
        ))
    return points


# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 컬렉션 생성 + upsert
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

def _create_collection(client: QdrantClient, name: str) -> None:
    if client.collection_exists(name):
        print(f"  ℹ️  {name} 이미 존재 — 삭제 후 재생성")
        client.delete_collection(name)

    client.create_collection(
        collection_name=name,
        vectors_config={
            "dense": VectorParams(size=DENSE_DIM, distance=Distance.COSINE),
        },
        sparse_vectors_config={
            "sparse": SparseVectorParams(),
        },
    )


def _upsert_batched(client: QdrantClient, name: str, points: list[PointStruct], batch_size: int = 256) -> None:
    total = len(points)
    for i in range(0, total, batch_size):
        batch = points[i : i + batch_size]
        client.upsert(collection_name=name, points=batch)
        print(f"  [{name}] {min(i + batch_size, total)}/{total} upsert 완료", end="\r")
    print()


def build_local_qdrant(recreate: bool = True) -> QdrantClient:
    """
    Colab 세션 안에서 로컬(파일 기반) Qdrant를 만들고, 업로드된 4+2개 파일로
    law_kb_jo_fixedid / law_kb_ho_fixedid 컬렉션을 채운다.
    반환된 client를 그대로 yoonha_rag_eval*.py의 QdrantClient 자리에 넣으면 된다.
    """
    client = QdrantClient(path=QDRANT_LOCAL_PATH)

    print("📦 jo 데이터 로드/병합 중...")
    merged_jo = _load_merged(VECTORS_JO, SPARSE_JO, CHUNKS_JO)
    print(f"  -> {len(merged_jo)}개 chunk 병합 완료")

    print("📦 ho 데이터 로드/병합 중...")
    merged_ho = _load_merged(VECTORS_HO, SPARSE_HO, CHUNKS_HO)
    print(f"  -> {len(merged_ho)}개 chunk 병합 완료")

    print(f"🗂️  {COLLECTION_JO} 컬렉션 생성...")
    _create_collection(client, COLLECTION_JO)
    _upsert_batched(client, COLLECTION_JO, _to_qdrant_points(merged_jo))

    print(f"🗂️  {COLLECTION_HO} 컬렉션 생성...")
    _create_collection(client, COLLECTION_HO)
    _upsert_batched(client, COLLECTION_HO, _to_qdrant_points(merged_ho))

    jo_count = client.count(COLLECTION_JO).count
    ho_count = client.count(COLLECTION_HO).count
    print(f"\n✅ 완료 — {COLLECTION_JO}: {jo_count}개, {COLLECTION_HO}: {ho_count}개")

    return client


if __name__ == "__main__":
    build_local_qdrant()


Writing /content/yoonha_colab_upsert.py


## 3. 업로드 파일 존재 확인

In [6]:
from pathlib import Path

REQUIRED_FILES = [
    "/content/vectors_jo_fixedid.npz",
    "/content/vectors_ho_fixedid.npz",
    "/content/sparse_weights_jo_fixedid.json",
    "/content/sparse_weights_ho_fixedid.json",
    "/content/chunks_jo_fixedid.json",
    "/content/chunks_ho_fixedid.json",
    "/content/gold_standard_v3.json",
]

missing = [f for f in REQUIRED_FILES if not Path(f).exists()]
if missing:
    print("⚠️  아직 안 올라온 파일:")
    for f in missing:
        print("  -", f)
    print("\n왼쪽 파일 탐색기(📁)에서 위 파일들을 /content/에 업로드한 뒤 이 셀을 다시 실행하세요.")
else:
    print("✅ 필요한 파일이 모두 있습니다.")


✅ 필요한 파일이 모두 있습니다.


## 4. 로컬 Qdrant 생성 + upsert

`/content/qdrant_local`에 파일 기반 Qdrant를 만들고,
`law_kb_jo_fixedid`(1186개), `law_kb_ho_fixedid`(7640개) 컬렉션을 채웁니다.
서버 프로세스가 따로 필요 없습니다.

In [7]:
import sys
sys.path.insert(0, "/content")

from yoonha_colab_upsert import build_local_qdrant

client = build_local_qdrant()

# 로컬(파일 기반) Qdrant는 경로 하나에 클라이언트가 동시에 하나만 붙을 수 있음.
# 아래 sweep 셀들은 !python으로 "별도 프로세스"에서 같은 경로를 다시 여니까,
# 여기서 만든 client를 계속 열어두면 그쪽에서 파일 락 충돌(AlreadyLocked)이 남.
# upsert가 끝났으면 바로 닫아서 락을 풀어준다.
client.close()
print("✅ upsert 완료, client 닫음 (sweep 프로세스가 같은 경로에 새로 붙을 수 있게)")


📦 jo 데이터 로드/병합 중...
  -> 1186개 chunk 병합 완료
📦 ho 데이터 로드/병합 중...
  -> 7640개 chunk 병합 완료
🗂️  law_kb_jo_fixedid 컬렉션 생성...
  [law_kb_jo_fixedid] 1186/1186 upsert 완료
🗂️  law_kb_ho_fixedid 컬렉션 생성...
  [law_kb_ho_fixedid] 7640/7640 upsert 완료

✅ 완료 — law_kb_jo_fixedid: 1186개, law_kb_ho_fixedid: 7640개
✅ upsert 완료, client 닫음 (sweep 프로세스가 같은 경로에 새로 붙을 수 있게)


## 5. 병렬(JoRAG/HoRAG) sweep 실행

`QDRANT_LOCAL_PATH`, `RERANKER_DEVICE` 환경변수만 잡아주면
`yoonha_rag_eval.py` 안의 client/reranker 생성 로직이 자동으로
Colab용(로컬 파일 기반 Qdrant + GPU)으로 분기합니다.

In [30]:
%%writefile /content/yoonha_rag_eval.py
"""
Workit - RAG 파라미터 sweep (alpha × reranking on/off, 캐싱 적용)
파일명: rag/yoonha_rag_eval.py

목적:
  JoRAG / HoRAG 각각에 대해 alpha(dense/sparse 비중)와 reranker1/reranker2
  사용 여부를 grid로 돌려서, 실버 스탠다드 평가셋 기준 Recall@1/5/10, MRR이
  가장 좋은 조합을 찾는다. (예: 기존에 검증된 "HoRAG alpha=0.6, K=10,
  Reranking OFF, MRR=0.9333"도 이런 sweep으로 나온 결과 — 이번엔 chunk_id
  체계가 fixedid로 바뀌고 ho가 목/세목까지 포함하게 됐으니 재검증 필요)

캐싱 (이번에 추가된 부분):
  - yoonha_law_rag.SweepCache 인스턴스를 sweep 시작 전에 딱 한 번 만들어서
    모든 조합(variant × alpha × rerank on/off)에 걸쳐 재사용한다.
  - alpha는 RRF 가중치 합산에만 쓰이므로, 임베딩과 Qdrant raw 검색 결과는
    (collection, query_text)당 한 번만 계산되고 alpha 5개 전체에서 재사용된다.
  - reranker 점수는 (reranker_name, query_text, chunk_id) 단위로 memoize되므로,
    같은 chunk가 여러 alpha/조합에서 다시 리랭킹 후보로 뽑혀도 한 번만 계산된다.
  - parent fetch / cross-ref 확장도 chunk_id -> payload 조회라 쿼리 간에도
    캐시가 재사용된다 (여러 조항이 같은 법 조항을 참조하는 경우가 많음).
  - 루프 순서(변수 조합이 바깥, 쿼리가 안쪽)는 그대로 둬도 된다 — 캐시가
    "먼저 계산된 값을 나중에 재사용"하는 방식이라 순서에 의존하지 않는다.

입력:
  - gold_standard_v3.json : [{"query_id", "query", "relevant_docs_jo": [...],
    "relevant_docs_ho": [...]}]
    (variant별로 정답 필드가 다름 — 아래 GT_FIELD_BY_VARIANT 참고)

출력:
  - eval_results.csv : 모든 조합의 Recall@1/5/10, MRR, 평균 소요시간
  - 콘솔에 최고 조합 + 캐시 히트 통계 출력

사용법:
    pip install qdrant-client FlagEmbedding transformers torch pandas
    python rag/yoonha_rag_eval.py
"""

from __future__ import annotations

import itertools
import json
import os
import time
from pathlib import Path

import pandas as pd
from qdrant_client import QdrantClient

from yoonha_law_rag import (
    load_embed_model,
    load_laws_ref,
    load_rerankers,
    search_jo,
    search_ho,
    SweepCache,
    QDRANT_HOST,
    QDRANT_PORT,
    DEFAULT_FETCH_K,
    DEFAULT_RERANK1_K,
    DEFAULT_RERANK2_K,
)

# Colab 등 로컬 서버(localhost:6333)에 못 붙는 환경에서는
# QDRANT_LOCAL_PATH 환경변수를 잡아주면 파일 기반 로컬 Qdrant를 쓴다
# (yoonha_colab_upsert.py가 만들어둔 QDRANT_LOCAL_PATH와 같은 경로를 넣으면 됨).
# GPU 리랭커를 쓰려면 RERANKER_DEVICE=cuda로 잡아준다 (기본값 cpu).
QDRANT_LOCAL_PATH_ENV = os.environ.get("QDRANT_LOCAL_PATH")
RERANKER_DEVICE = os.environ.get("RERANKER_DEVICE", "cpu")


def get_qdrant_client() -> QdrantClient:
    if QDRANT_LOCAL_PATH_ENV:
        print(f"🗄️  로컬(파일 기반) Qdrant 사용: {QDRANT_LOCAL_PATH_ENV}")
        return QdrantClient(path=QDRANT_LOCAL_PATH_ENV)
    print(f"🗄️  서버 Qdrant 사용: {QDRANT_HOST}:{QDRANT_PORT}")
    return QdrantClient(host=QDRANT_HOST, port=QDRANT_PORT)

_THIS_DIR = Path(__file__).resolve().parent
GOLD_STANDARD_PATH = Path("/content/gold_standard_v3.json")  # Colab: 평평한 경로
RESULTS_CSV = Path("/content/eval_results.csv")

# sweep 그리드 — 필요하면 여기만 수정
ALPHA_GRID = [0.3, 0.5, 0.6, 0.7, 1.0]           # 1.0 = dense only
RERANK_GRID = [
    # (use_reranker1, use_reranker2)
    (False, False),   # reranking 전체 OFF
    (True,  False),   # 1단계만
    (False, True),    # 2단계만
    (True,  True),    # 둘 다 ON
]
VARIANTS = ["jo", "ho"]  # JoRAG / HoRAG 둘 다 sweep

# gold_standard_v3.json은 variant별로 정답 필드가 다르다.
#   - JoRAG는 항상 조 단위로 직접 검색하므로 relevant_docs_jo와 비교
#   - HoRAG는 chunk_id를 호/목/세목 단위 그대로 반환하므로(parent fetch는
#     text만 교체하고 chunk_id는 안 바꿈) relevant_docs_ho와 비교
GT_FIELD_BY_VARIANT = {
    "jo": "relevant_docs_jo",
    "ho": "relevant_docs_ho",
}

# 주의: fetch_k/rerank1_k/rerank2_k를 sweep 그리드에 추가하고 싶다면
# SweepCache의 raw_search 캐시 키에 fetch_k가 이미 포함돼 있어 안전하지만,
# rerank_score 캐시는 rerank1_k/rerank2_k와 무관하게 chunk 단위로 캐시되므로
# 그 자체로는 문제 없다 (top_k 절단만 나중에 다시 적용됨).


def load_gold_standard(path: Path = GOLD_STANDARD_PATH) -> list[dict]:
    with open(path, encoding="utf-8") as f:
        return json.load(f)


def evaluate_combo(
    variant       : str,
    client        : QdrantClient,
    model,
    laws_ref      : dict,
    reranker1,
    reranker2,
    use_reranker1 : bool,
    use_reranker2 : bool,
    alpha         : float,
    gold_standard : list[dict],
    cache         : SweepCache,
    top_k_eval    : int = 10,
) -> dict:
    """단일 조합(variant, alpha, reranker on/off)에 대해 Recall@1/5/10, MRR 계산."""
    recall1 = recall5 = recall10 = mrr = 0
    n = len(gold_standard)
    t0 = time.time()

    search_fn = search_jo if variant == "jo" else search_ho
    # variant(jo/ho)에 맞는 정답 필드를 미리 골라둔다.
    gt_field = GT_FIELD_BY_VARIANT[variant]

    for item in gold_standard:
        gt_docs = set(item[gt_field])

        kwargs = dict(
            clause_text=item["query"],
            client=client,
            model=model,
            laws_ref=laws_ref,
            reranker1=reranker1,
            reranker2=reranker2,
            use_reranker1=use_reranker1,
            use_reranker2=use_reranker2,
            top_k=top_k_eval,
            alpha=alpha,
            fetch_k=DEFAULT_FETCH_K,
            rerank1_k=DEFAULT_RERANK1_K,
            rerank2_k=DEFAULT_RERANK2_K,
            cache=cache,
        )
        law_refs = search_fn(**kwargs)
        ranked_ids = [r.chunk_id for r in law_refs]

        if any(d in gt_docs for d in ranked_ids[:1]):  recall1  += 1
        if any(d in gt_docs for d in ranked_ids[:5]):  recall5  += 1
        if any(d in gt_docs for d in ranked_ids[:10]): recall10 += 1
        for rank, d in enumerate(ranked_ids, 1):
            if d in gt_docs:
                mrr += 1 / rank
                break

    elapsed = time.time() - t0

    return {
        "variant":        variant,
        "alpha":          alpha,
        "use_reranker1":  use_reranker1,
        "use_reranker2":  use_reranker2,
        "Recall@1":       round(recall1  / n, 4),
        "Recall@5":       round(recall5  / n, 4),
        "Recall@10":      round(recall10 / n, 4),
        "MRR":            round(mrr      / n, 4),
        "avg_sec_per_query": round(elapsed / n, 3),
    }


def main():
    client   = get_qdrant_client()
    model    = load_embed_model()
    laws_ref = load_laws_ref()
    gold_standard = load_gold_standard()

    print(f"실버 스탠다드 {len(gold_standard)}개 로드 완료")

    # reranker는 무거우니 한 번만 로드 (grid 안에서 껐다 켰다는 플래그로만 토글)
    reranker1, reranker2 = load_rerankers(device=RERANKER_DEVICE)

    # sweep 전체(40조합)에서 재사용할 캐시 — combo/쿼리 루프 밖에서 딱 한 번만 생성
    cache = SweepCache()

    all_results = []
    combos = list(itertools.product(VARIANTS, ALPHA_GRID, RERANK_GRID))
    print(f"총 {len(combos)}개 조합 sweep 시작...\n")

    t_sweep_start = time.time()

    for i, (variant, alpha, (use_r1, use_r2)) in enumerate(combos, 1):
        print(f"[{i}/{len(combos)}] variant={variant} alpha={alpha} "
              f"reranker1={use_r1} reranker2={use_r2} ...")

        result = evaluate_combo(
            variant, client, model, laws_ref, reranker1, reranker2,
            use_r1, use_r2, alpha, gold_standard, cache,
        )
        all_results.append(result)
        print(f"    -> Recall@10={result['Recall@10']} MRR={result['MRR']} "
              f"({result['avg_sec_per_query']}초/쿼리) | 캐시: {cache.stats()}\n")

    t_sweep_total = time.time() - t_sweep_start

    df = pd.DataFrame(all_results)
    df = df.sort_values(["variant", "MRR"], ascending=[True, False])
    df.to_csv(RESULTS_CSV, index=False)
    print(f"\n✅ 전체 결과 저장: {RESULTS_CSV}")
    print(f"⏱️  전체 sweep 소요 시간: {t_sweep_total:.1f}초 "
          f"({t_sweep_total/60:.1f}분)")
    print(f"📦 최종 캐시 통계: {cache.stats()}")

    print("\n=== variant별 최고 조합 (MRR 기준) ===")
    for variant in VARIANTS:
        best = df[df["variant"] == variant].iloc[0]
        print(f"[{variant}] alpha={best['alpha']}, reranker1={best['use_reranker1']}, "
              f"reranker2={best['use_reranker2']} "
              f"-> Recall@10={best['Recall@10']}, MRR={best['MRR']}")


if __name__ == "__main__":
    main()


Overwriting /content/yoonha_rag_eval.py


In [32]:
import os

os.environ["QDRANT_LOCAL_PATH"] = "/content/qdrant_local"
os.environ["RERANKER_DEVICE"] = "cuda"

!python /content/yoonha_rag_eval.py


🗄️  로컬(파일 기반) Qdrant 사용: /content/qdrant_local
📦 임베딩 모델 로드: BAAI/bge-m3
Fetching 30 files: 100% 30/30 [00:00<00:00, 4336.99it/s]
Download complete: : 0.00B [00:00, ?B/s]
Loading weights: 100% 391/391 [00:08<00:00, 47.51it/s]
  ⚠️  laws_ref.json 없음: /data/hn_seed/law_refs.json
실버 스탠다드 50개 로드 완료
📦 Re-ranker 1단계 로드: Dongjin-kr/ko-reranker
Loading weights: 100% 393/393 [00:00<00:00, 1507.60it/s]
📦 Re-ranker 2단계 로드: BAAI/bge-reranker-v2-m3
Loading weights: 100% 393/393 [00:00<00:00, 978.51it/s]
총 40개 조합 sweep 시작...

[1/40] variant=jo alpha=0.3 reranker1=False reranker2=False ...
    -> Recall@10=0.56 MRR=0.4326 (0.161초/쿼리) | 캐시: {'embed_cached': 50, 'raw_search_cached': 50, 'scroll_cached': 0, 'rerank_cached': 0}

[2/40] variant=jo alpha=0.3 reranker1=True reranker2=False ...
Traceback (most recent call last):
  File "/content/yoonha_rag_eval.py", line 232, in <module>
    main()
  File "/content/yoonha_rag_eval.py", line 205, in main
    result = evaluate_combo(
             ^^^^^^^^^^^^^^

In [16]:
import pandas as pd

df_parallel = pd.read_csv("/content/eval_results.csv")
df_parallel.sort_values(["variant", "MRR"], ascending=[True, False]).head(10)


,variant,alpha,use_reranker1,use_reranker2,Recall@1,Recall@5,Recall@10,MRR,avg_sec_per_query
0,ho,0.3,True,False,0.56,0.88,0.92,0.6985,2.582
1,ho,0.5,True,False,0.56,0.88,0.92,0.6985,0.172
2,ho,0.6,True,False,0.56,0.88,0.92,0.6985,0.152
3,ho,0.7,True,False,0.56,0.88,0.92,0.6985,0.154
4,ho,1.0,True,False,0.56,0.88,0.92,0.6985,0.153
5,ho,0.3,True,True,0.56,0.90,0.94,0.6848,0.178
6,ho,0.5,True,True,0.56,0.90,0.94,0.6848,0.155
7,ho,0.6,True,True,0.56,0.90,0.94,0.6848,0.154
8,ho,0.7,True,True,0.56,0.90,0.94,0.6848,0.152
9,ho,1.0,True,True,0.56,0.90,0.94,0.6848,0.158


## 6. 계층형(Cascaded) sweep 실행

1단계(JoRAG top-k)로 후보 조를 추린 뒤, 그 안에서만 2단계(HoRAG) 검색을
하는 계층형 variant입니다. `stage1_recall` 컬럼으로 1단계 필터링 자체가
병목인지 진단할 수 있습니다.

In [20]:
%%writefile /content/yoonha_rag_eval_cascaded.py
"""
Workit - 계층형(Cascaded) RAG 파라미터 sweep
파일명: rag/yoonha_rag_eval_cascaded.py

목적:
  yoonha_rag_eval.py(병렬 JoRAG/HoRAG 독립 검색)와는 별개로, 계층형
  (1단계 JoRAG top-k → 2단계 그 조로 제한된 HoRAG) 구조의 최적 조합을 찾는다.

  병렬 버전과 다른 점:
    - variant 축이 없다 (jo 단독은 계층형 개념이 성립하지 않음 — 계층형은
      항상 "jo top-k → ho 제한 검색" 전체 파이프라인 단위로만 평가한다).
    - jo_top_k가 새 sweep 축으로 추가된다. 1단계 recall이 2단계 recall의
      상한선이 되므로, jo_top_k가 너무 작으면 아무리 alpha/reranker를
      잘 조합해도 최종 성능이 낮게 나온다.
    - 각 조합마다 stage1_recall(1단계에서 정답 조가 top-k 안에 살아남은
      비율)을 별도로 기록한다. 최종 MRR이 낮게 나왔을 때 "1단계 필터링
      자체가 문제였는지" "2단계 검색/리랭킹이 문제였는지"를 구분하기 위함.

  주의 — grid 크기:
    alpha 5개 × rerank 조합 4개 × jo_top_k 5개(10/20/30/40/50) = 100조합.
    jo_top_k=30까지는 이미 확인했고(최고 MRR 0.7012, 병렬 HoRAG 0.6985과 사실상
    동률), 40/50에서도 우위가 없으면 계층형 구조는 폐기하는 쪽으로 결론 낸다.
    필요하면 ALPHA_GRID나 RERANK_GRID를 줄여서 조합 수를 다시 낮춰도 된다
    (jo_top_k만 우선 넓게 보고 싶으면, 예: RERANK_GRID를 최고 조합이었던
    (True, False) 하나로만 좁혀서 alpha 5 × jo_top_k 5 = 25조합으로도 충분).

캐싱:
  yoonha_law_rag.SweepCache를 sweep 시작 전에 한 번만 생성해서 전체 조합에서
  재사용한다. 다만 계층형은 jo_top_k가 바뀌면 2단계 검색의 Qdrant 필터
  자체가 달라지므로, raw_search 캐시 재사용률은 병렬 버전만큼 높지 않다
  (필터 지문이 캐시 키에 포함되어 있어서 서로 다른 필터끼리 섞이진 않음 —
  yoonha_law_rag_cascaded._hybrid_search_ho_filtered 참고). 반면 임베딩
  캐시와 parent/cross-ref scroll 캐시는 jo_top_k와 무관하게 그대로
  재사용된다.

입력/출력은 yoonha_rag_eval.py와 동일한 gold_standard_v3.json,
eval_results_cascaded.csv (파일명만 다름, 기존 결과를 덮어쓰지 않음).
"""

from __future__ import annotations

import itertools
import json
import os
import time
from pathlib import Path

import pandas as pd
from qdrant_client import QdrantClient

from yoonha_law_rag import (
    load_embed_model,
    load_laws_ref,
    load_rerankers,
    SweepCache,
    QDRANT_HOST,
    QDRANT_PORT,
    DEFAULT_FETCH_K,
    DEFAULT_RERANK1_K,
    DEFAULT_RERANK2_K,
    derive_jo_id,
)
from yoonha_law_rag_cascaded import (
    search_cascaded,
    get_stage1_jo_candidates,
    DEFAULT_JO_TOP_K,
)

# Colab 등 로컬 서버(localhost:6333)에 못 붙는 환경에서는
# QDRANT_LOCAL_PATH 환경변수를 잡아주면 파일 기반 로컬 Qdrant를 쓴다
# (yoonha_colab_upsert.py가 만들어둔 QDRANT_LOCAL_PATH와 같은 경로를 넣으면 됨).
# GPU 리랭커를 쓰려면 RERANKER_DEVICE=cuda로 잡아준다 (기본값 cpu).
QDRANT_LOCAL_PATH_ENV = os.environ.get("QDRANT_LOCAL_PATH")
RERANKER_DEVICE = os.environ.get("RERANKER_DEVICE", "cpu")


def get_qdrant_client() -> QdrantClient:
    if QDRANT_LOCAL_PATH_ENV:
        print(f"🗄️  로컬(파일 기반) Qdrant 사용: {QDRANT_LOCAL_PATH_ENV}")
        return QdrantClient(path=QDRANT_LOCAL_PATH_ENV)
    print(f"🗄️  서버 Qdrant 사용: {QDRANT_HOST}:{QDRANT_PORT}")
    return QdrantClient(host=QDRANT_HOST, port=QDRANT_PORT)

_THIS_DIR = Path(__file__).resolve().parent
GOLD_STANDARD_PATH = Path("/content/gold_standard_v3.json")  # Colab: 평평한 경로
RESULTS_CSV = Path("/content/eval_results_cascaded.csv")

# sweep 그리드 — 필요하면 여기만 수정
ALPHA_GRID   = [0.3, 0.5, 0.6, 0.7, 1.0]           # 1.0 = dense only
JO_TOP_K_GRID = [10, 20, 30, 40, 50]                # 1단계(조 단위)에서 남길 후보 수
RERANK_GRID = [
    # (use_reranker1, use_reranker2)
    (False, False),   # reranking 전체 OFF
    (True,  False),   # 1단계만
    (False, True),    # 2단계만
    (True,  True),    # 둘 다 ON
]

# 계층형은 항상 relevant_docs_ho와 비교 (최종 출력은 조 텍스트로 통일되지만
# chunk_id 자체는 ho-level 그대로 반환 — 병렬 버전의 HoRAG와 동일한 규칙)
GT_FIELD = "relevant_docs_ho"


def load_gold_standard(path: Path = GOLD_STANDARD_PATH) -> list[dict]:
    with open(path, encoding="utf-8") as f:
        return json.load(f)


def _get_ho_parent_id(chunk_id: str) -> str:
    """
    ho-level chunk_id에서 그 조에 해당하는 jo_id를 역산한다 (stage1_recall 진단용).

    이전 버전은 payload의 parent_chunk_id 필드를 Qdrant scroll로 조회했었는데,
    실제 데이터 검증 결과 그 필드는 JO 컬렉션이 아니라 HO 컬렉션 자기 자신을
    가리키고 있어서 JO id와 절대 일치하지 않았다 (그래서 stage1_recall이 항상
    0.0으로 나왔음). derive_jo_id()는 chunk_id 문자열 자체에서 직접 역산하므로
    Qdrant 조회 자체가 필요 없다 — 훨씬 빠르고 정확하다.
    """
    return derive_jo_id(chunk_id)


def evaluate_combo_cascaded(
    client        : QdrantClient,
    model,
    laws_ref      : dict,
    reranker1,
    reranker2,
    use_reranker1 : bool,
    use_reranker2 : bool,
    alpha         : float,
    jo_top_k      : int,
    gold_standard : list[dict],
    cache         : SweepCache,
    top_k_eval    : int = 10,
) -> dict:
    """단일 조합(alpha, jo_top_k, reranker on/off)에 대해 Recall@1/5/10, MRR, stage1_recall 계산."""
    recall1 = recall5 = recall10 = mrr = 0
    stage1_hits = stage1_total = 0
    n = len(gold_standard)
    t0 = time.time()

    for item in gold_standard:
        gt_docs = set(item[GT_FIELD])
        query   = item["query"]

        # --- stage1 진단: 정답 ho chunk의 parent 조가 1단계 top-k 안에 살아남았는지 ---
        stage1_candidates = get_stage1_jo_candidates(
            query, client, model, alpha=alpha, fetch_k=DEFAULT_FETCH_K,
            jo_top_k=jo_top_k, cache=cache,
        )
        gt_parents = set()
        for d in gt_docs:
            pid = _get_ho_parent_id(d)
            if pid:
                gt_parents.add(pid)
        if gt_parents:
            stage1_total += 1
            if gt_parents & set(stage1_candidates):
                stage1_hits += 1

        # --- 실제 최종 검색 (1단계+2단계 전체) ---
        law_refs = search_cascaded(
            clause_text=query,
            client=client,
            model=model,
            laws_ref=laws_ref,
            reranker1=reranker1,
            reranker2=reranker2,
            use_reranker1=use_reranker1,
            use_reranker2=use_reranker2,
            top_k=top_k_eval,
            alpha=alpha,
            fetch_k=DEFAULT_FETCH_K,
            rerank1_k=DEFAULT_RERANK1_K,
            rerank2_k=DEFAULT_RERANK2_K,
            jo_top_k=jo_top_k,
            cache=cache,
        )
        ranked_ids = [r.chunk_id for r in law_refs]

        if any(d in gt_docs for d in ranked_ids[:1]):  recall1  += 1
        if any(d in gt_docs for d in ranked_ids[:5]):  recall5  += 1
        if any(d in gt_docs for d in ranked_ids[:10]): recall10 += 1
        for rank, d in enumerate(ranked_ids, 1):
            if d in gt_docs:
                mrr += 1 / rank
                break

    elapsed = time.time() - t0

    return {
        "alpha":            alpha,
        "jo_top_k":         jo_top_k,
        "use_reranker1":    use_reranker1,
        "use_reranker2":    use_reranker2,
        "Recall@1":         round(recall1  / n, 4),
        "Recall@5":         round(recall5  / n, 4),
        "Recall@10":        round(recall10 / n, 4),
        "MRR":              round(mrr      / n, 4),
        "stage1_recall":    round(stage1_hits / stage1_total, 4) if stage1_total else None,
        "avg_sec_per_query": round(elapsed / n, 3),
    }


def main():
    client   = get_qdrant_client()
    model    = load_embed_model()
    laws_ref = load_laws_ref()
    gold_standard = load_gold_standard()

    print(f"실버 스탠다드 {len(gold_standard)}개 로드 완료")

    reranker1, reranker2 = load_rerankers(device=RERANKER_DEVICE)

    # sweep 전체에서 재사용할 캐시 — combo/쿼리 루프 밖에서 딱 한 번만 생성.
    # yoonha_rag_eval.py(병렬 버전)의 캐시와는 별개 인스턴스 — 두 sweep을
    # 동시에 돌리더라도 서로 캐시가 섞이지 않는다.
    cache = SweepCache()

    all_results = []
    combos = list(itertools.product(ALPHA_GRID, JO_TOP_K_GRID, RERANK_GRID))
    print(f"총 {len(combos)}개 조합(계층형) sweep 시작...\n")

    t_sweep_start = time.time()

    for i, (alpha, jo_top_k, (use_r1, use_r2)) in enumerate(combos, 1):
        print(f"[{i}/{len(combos)}] alpha={alpha} jo_top_k={jo_top_k} "
              f"reranker1={use_r1} reranker2={use_r2} ...")

        result = evaluate_combo_cascaded(
            client, model, laws_ref, reranker1, reranker2,
            use_r1, use_r2, alpha, jo_top_k, gold_standard, cache,
        )
        all_results.append(result)
        print(f"    -> Recall@10={result['Recall@10']} MRR={result['MRR']} "
              f"stage1_recall={result['stage1_recall']} "
              f"({result['avg_sec_per_query']}초/쿼리) | 캐시: {cache.stats()}\n")

    t_sweep_total = time.time() - t_sweep_start

    df = pd.DataFrame(all_results)
    df = df.sort_values("MRR", ascending=False)
    df.to_csv(RESULTS_CSV, index=False)
    print(f"\n✅ 전체 결과 저장: {RESULTS_CSV}")
    print(f"⏱️  전체 sweep 소요 시간: {t_sweep_total:.1f}초 "
          f"({t_sweep_total/60:.1f}분)")
    print(f"📦 최종 캐시 통계: {cache.stats()}")

    print("\n=== 계층형 최고 조합 (MRR 기준) ===")
    best = df.iloc[0]
    print(f"alpha={best['alpha']}, jo_top_k={best['jo_top_k']}, "
          f"reranker1={best['use_reranker1']}, reranker2={best['use_reranker2']} "
          f"-> Recall@10={best['Recall@10']}, MRR={best['MRR']}, "
          f"stage1_recall={best['stage1_recall']}")

    print("\n※ 병렬 버전(eval_results.csv)의 HoRAG 최고 MRR과 이 결과를 나란히")
    print("   비교해서, 계층형이 실제로 이득인지 판단하시면 됩니다.")


if __name__ == "__main__":
    main()

Overwriting /content/yoonha_rag_eval_cascaded.py


In [21]:
import os
os.environ["QDRANT_LOCAL_PATH"] = "/content/qdrant_local"
os.environ["RERANKER_DEVICE"] = "cuda"

In [22]:
!python /content/yoonha_rag_eval_cascaded.py


🗄️  로컬(파일 기반) Qdrant 사용: /content/qdrant_local
📦 임베딩 모델 로드: BAAI/bge-m3
Fetching 30 files: 100% 30/30 [00:00<00:00, 12805.73it/s]
Download complete: : 0.00B [00:00, ?B/s]
Loading weights: 100% 391/391 [00:08<00:00, 47.62it/s]
  ⚠️  laws_ref.json 없음: /data/hn_seed/law_refs.json
실버 스탠다드 50개 로드 완료
📦 Re-ranker 1단계 로드: Dongjin-kr/ko-reranker
Loading weights: 100% 393/393 [00:00<00:00, 1344.86it/s]
📦 Re-ranker 2단계 로드: BAAI/bge-reranker-v2-m3
Loading weights: 100% 393/393 [00:00<00:00, 1087.21it/s]
총 100개 조합(계층형) sweep 시작...

[1/100] alpha=0.3 jo_top_k=10 reranker1=False reranker2=False ...
    -> Recall@10=0.48 MRR=0.3463 stage1_recall=0.56 (1.317초/쿼리) | 캐시: {'embed_cached': 50, 'raw_search_cached': 100, 'scroll_cached': 767, 'rerank_cached': 0}

[2/100] alpha=0.3 jo_top_k=10 reranker1=True reranker2=False ...
    -> Recall@10=0.5 MRR=0.3713 stage1_recall=0.56 (0.488초/쿼리) | 캐시: {'embed_cached': 50, 'raw_search_cached': 100, 'scroll_cached': 773, 'rerank_cached': 867}

[3/100] alpha=0.3 jo_to

In [23]:
import pandas as pd

df_cascaded = pd.read_csv("/content/eval_results_cascaded.csv")
df_cascaded.sort_values("MRR", ascending=False).head(10)


,alpha,jo_top_k,use_reranker1,use_reranker2,Recall@1,Recall@5,Recall@10,MRR,stage1_recall,avg_sec_per_query
0,1.0,40,True,False,0.58,0.88,0.92,0.7179,1.00,0.131
1,1.0,50,True,False,0.58,0.88,0.92,0.7179,1.00,0.135
2,0.5,30,True,False,0.56,0.86,0.90,0.7012,0.98,0.114
3,0.6,30,True,False,0.56,0.86,0.90,0.7012,0.98,0.125
4,0.7,30,True,False,0.56,0.86,0.90,0.7012,0.98,0.115
5,1.0,30,True,False,0.56,0.86,0.90,0.7012,0.98,0.101
6,0.5,50,True,False,0.56,0.86,0.90,0.6979,0.98,0.147
7,0.6,50,True,False,0.56,0.86,0.90,0.6979,0.98,0.155
8,0.6,40,True,False,0.56,0.86,0.90,0.6979,0.98,0.125
9,0.3,40,True,False,0.56,0.86,0.90,0.6979,0.98,0.327


## 7. 병렬 vs 계층형 비교

병렬 HoRAG의 최고 MRR과 계층형의 최고 MRR을 나란히 놓고 비교합니다.
계층형이 더 낮게 나온다면 `stage1_recall`을 같이 봐서, 1단계 필터링에서
정답 조가 얼마나 손실됐는지 먼저 확인하세요.

In [24]:
best_parallel_ho = df_parallel[df_parallel["variant"] == "ho"].sort_values("MRR", ascending=False).iloc[0]
best_cascaded = df_cascaded.sort_values("MRR", ascending=False).iloc[0]

print("=== 병렬 HoRAG 최고 조합 ===")
print(best_parallel_ho[["alpha", "use_reranker1", "use_reranker2", "Recall@10", "MRR"]])

print("\n=== 계층형 최고 조합 ===")
print(best_cascaded[["alpha", "jo_top_k", "use_reranker1", "use_reranker2", "Recall@10", "MRR", "stage1_recall"]])


=== 병렬 HoRAG 최고 조합 ===
alpha               0.3
use_reranker1      True
use_reranker2     False
Recall@10          0.92
MRR              0.6985
Name: 0, dtype: object

=== 계층형 최고 조합 ===
alpha               1.0
jo_top_k             40
use_reranker1      True
use_reranker2     False
Recall@10          0.92
MRR              0.7179
stage1_recall       1.0
Name: 0, dtype: object


In [25]:
from google.colab import files

files.download("/content/eval_results.csv")
files.download("/content/eval_results_cascaded.csv")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## 8. 쿼리 단위 오답 분석

`horag_r1_only` vs `horag_r1_r2`, `cascaded_a1_k30` vs `cascaded_a1_k40` 두 쌍을 비교해서
어떤 쿼리에서 순위가 바뀌었는지 확인합니다. 필요하면 스크립트 안 `COMBOS`/`DIFF_PAIRS`를 고쳐서
다른 조합도 비교할 수 있습니다.

In [28]:
%%writefile /content/yoonha_rag_query_diff.py
"""
Workit - 쿼리 단위 결과 비교 스크립트
파일명: rag/yoonha_rag_query_diff.py

목적:
  지금까지 쓴 yoonha_rag_eval.py / yoonha_rag_eval_cascaded.py는 조합별
  집계 지표(Recall@1/5/10, MRR)만 CSV에 남기고 "쿼리 하나하나가 어떻게
  검색됐는지"는 버린다. 그래서 아래 두 가지를 확인하려면 쿼리 단위 결과가
  따로 필요하다:

    1) HoRAG reranker1=True 단독 vs reranker1+reranker2 둘 다 켰을 때,
       어떤 쿼리에서 순위가 나빠지는지 (MRR 0.6985 → 0.6848 하락의 원인)
    2) 계층형 alpha=1.0, jo_top_k=30 vs jo_top_k=40일 때,
       어떤 쿼리가 바뀌어서 MRR 0.7012 → 0.7179가 됐는지
       (진짜 개선인지, 우연히 쿼리 1개가 뒤집힌 건지 확인용)

이 스크립트는 COMBOS에 정의된 조합들을 실제로 실행해서 쿼리 50개 각각의
순위 결과를 저장하고, DIFF_PAIRS에 정의된 조합 쌍끼리 RR(reciprocal rank)이
달라진 쿼리만 추려서 사람이 읽을 수 있는 형태로 출력한다.

출력:
  - query_level_results.csv : 모든 조합 × 모든 쿼리의 순위 결과 (long format)
  - diff_<combo1>_vs_<combo2>.csv : 두 조합 사이에 RR이 달라진 쿼리만 모은 diff
  - 콘솔에 diff 쿼리들을 사람이 읽기 좋은 형태로 출력 (질의문 + 정답 + 검색된
    상위 3개 + 실제 조문 텍스트 스니펫)

사용법:
    python rag/yoonha_rag_query_diff.py
"""

from __future__ import annotations

import json
import os
from dataclasses import dataclass, field
from pathlib import Path

import pandas as pd
from qdrant_client import QdrantClient
from qdrant_client.models import Filter, FieldCondition, MatchValue

from yoonha_law_rag import (
    load_embed_model,
    load_laws_ref,
    load_rerankers,
    search_ho,
    SweepCache,
    QDRANT_HOST,
    QDRANT_PORT,
    DEFAULT_FETCH_K,
    DEFAULT_RERANK1_K,
    DEFAULT_RERANK2_K,
    COLLECTION_HO,
)
from yoonha_law_rag_cascaded import search_cascaded, DEFAULT_JO_TOP_K

# Colab 등에서는 yoonha_rag_eval*.py와 동일한 방식으로 환경변수로 분기한다.
QDRANT_LOCAL_PATH_ENV = os.environ.get("QDRANT_LOCAL_PATH")
RERANKER_DEVICE = os.environ.get("RERANKER_DEVICE", "cpu")


def get_qdrant_client() -> QdrantClient:
    if QDRANT_LOCAL_PATH_ENV:
        print(f"🗄️  로컬(파일 기반) Qdrant 사용: {QDRANT_LOCAL_PATH_ENV}")
        return QdrantClient(path=QDRANT_LOCAL_PATH_ENV)
    print(f"🗄️  서버 Qdrant 사용: {QDRANT_HOST}:{QDRANT_PORT}")
    return QdrantClient(host=QDRANT_HOST, port=QDRANT_PORT)


_THIS_DIR = Path(__file__).resolve().parent
GOLD_STANDARD_PATH = Path("/content/gold_standard_v3.json") if QDRANT_LOCAL_PATH_ENV else (
    _THIS_DIR.parent / "rag" / "evaluation" / "gold_standard_v3.json"
)
OUT_DIR = Path("/content") if QDRANT_LOCAL_PATH_ENV else _THIS_DIR

TOP_K_EVAL = 10

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 비교하고 싶은 조합들 — 필요하면 여기만 수정
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

@dataclass
class ComboSpec:
    name: str
    kind: str  # "ho" | "cascaded"
    alpha: float
    use_reranker1: bool
    use_reranker2: bool
    jo_top_k: int = DEFAULT_JO_TOP_K  # kind="cascaded"일 때만 사용


COMBOS: list[ComboSpec] = [
    # --- ① HoRAG: reranker1 단독 vs reranker1+reranker2 ---
    ComboSpec("horag_r1_only", "ho", alpha=0.6, use_reranker1=True, use_reranker2=False),
    ComboSpec("horag_r1_r2",   "ho", alpha=0.6, use_reranker1=True, use_reranker2=True),
    # --- ② 계층형: alpha=1.0, jo_top_k=30 vs 40 ---
    ComboSpec("cascaded_a1_k30", "cascaded", alpha=1.0, use_reranker1=True, use_reranker2=False, jo_top_k=30),
    ComboSpec("cascaded_a1_k40", "cascaded", alpha=1.0, use_reranker1=True, use_reranker2=False, jo_top_k=40),
]

# 어떤 조합끼리 diff를 뽑을지 — (combo1_name, combo2_name) 튜플 리스트
DIFF_PAIRS: list[tuple[str, str]] = [
    ("horag_r1_only", "horag_r1_r2"),
    ("cascaded_a1_k30", "cascaded_a1_k40"),
]

GT_FIELD = "relevant_docs_ho"


def load_gold_standard(path: Path = GOLD_STANDARD_PATH) -> list[dict]:
    with open(path, encoding="utf-8") as f:
        return json.load(f)


def _lookup_text(chunk_id: str, client: QdrantClient, cache: SweepCache, snippet_len: int = 70) -> str:
    """chunk_id의 실제 조문 텍스트 앞부분을 조회 (diff 결과를 사람이 읽기 위한 용도)."""
    cache_key = (COLLECTION_HO, chunk_id)
    if cache_key in cache.scroll:
        payload = cache.scroll[cache_key]
    else:
        results = client.scroll(
            collection_name=COLLECTION_HO,
            scroll_filter=Filter(
                must=[FieldCondition(key="chunk_id", match=MatchValue(value=chunk_id))]
            ),
            limit=1,
            with_payload=True,
            with_vectors=False,
        )
        points = results[0]
        if not points:
            return "(텍스트 없음)"
        payload = points[0].payload
        cache.scroll[cache_key] = payload

    text = payload.get("text", payload.get("chunk_text", ""))
    return text[:snippet_len].replace("\n", " ") + ("…" if len(text) > snippet_len else "")


def run_combo(
    combo: ComboSpec,
    client: QdrantClient,
    model,
    laws_ref: dict,
    reranker1,
    reranker2,
    gold_standard: list[dict],
    cache: SweepCache,
) -> list[dict]:
    """단일 조합에 대해 쿼리 50개 각각의 순위 결과를 담은 리스트를 반환."""
    rows = []
    search_fn = search_ho if combo.kind == "ho" else search_cascaded

    for item in gold_standard:
        gt_docs = set(item[GT_FIELD])
        query = item["query"]

        kwargs = dict(
            clause_text=query,
            client=client,
            model=model,
            laws_ref=laws_ref,
            reranker1=reranker1,
            reranker2=reranker2,
            use_reranker1=combo.use_reranker1,
            use_reranker2=combo.use_reranker2,
            top_k=TOP_K_EVAL,
            alpha=combo.alpha,
            fetch_k=DEFAULT_FETCH_K,
            rerank1_k=DEFAULT_RERANK1_K,
            rerank2_k=DEFAULT_RERANK2_K,
            cache=cache,
        )
        if combo.kind == "cascaded":
            kwargs["jo_top_k"] = combo.jo_top_k

        law_refs = search_fn(**kwargs)
        ranked_ids = [r.chunk_id for r in law_refs]

        rank_of_hit = None
        for rank, cid in enumerate(ranked_ids, 1):
            if cid in gt_docs:
                rank_of_hit = rank
                break
        rr = round(1 / rank_of_hit, 4) if rank_of_hit else 0.0

        rows.append({
            "combo": combo.name,
            "query_id": item.get("query_id", ""),
            "query": query,
            "gt_docs": list(gt_docs),
            "rank_of_hit": rank_of_hit,
            "rr": rr,
            "hit@1": bool(rank_of_hit == 1),
            "hit@5": bool(rank_of_hit is not None and rank_of_hit <= 5),
            "hit@10": bool(rank_of_hit is not None and rank_of_hit <= 10),
            "ranked_ids_top5": ranked_ids[:5],
        })

    return rows


def print_and_save_diff(
    df: pd.DataFrame,
    combo1: str,
    combo2: str,
    client: QdrantClient,
    cache: SweepCache,
) -> pd.DataFrame:
    """combo1 -> combo2 사이에 RR이 달라진 쿼리만 골라서 사람이 읽을 수 있게 출력하고 CSV로 저장."""
    a = df[df.combo == combo1].set_index("query_id")
    b = df[df.combo == combo2].set_index("query_id")

    merged = a[["query", "gt_docs", "rr", "rank_of_hit", "ranked_ids_top5"]].join(
        b[["rr", "rank_of_hit", "ranked_ids_top5"]], lsuffix="_1", rsuffix="_2"
    )
    merged["rr_delta"] = merged["rr_2"] - merged["rr_1"]
    changed = merged[merged["rr_delta"] != 0].sort_values("rr_delta")

    print(f"\n{'='*70}")
    print(f"  {combo1}  vs  {combo2}")
    print(f"  총 {len(merged)}개 쿼리 중 {len(changed)}개 쿼리에서 순위 변화 발생")
    print(f"{'='*70}")

    for qid, row in changed.iterrows():
        direction = "🔺 개선" if row["rr_delta"] > 0 else "🔻 악화"
        print(f"\n[{qid}] {direction}  RR: {row['rr_1']} → {row['rr_2']}")
        print(f"  질의: {row['query'][:80]}")
        print(f"  정답: {row['gt_docs']}")

        print(f"  [{combo1}] rank={row['rank_of_hit_1']}  top3:")
        for cid in row["ranked_ids_top5_1"][:3]:
            mark = "✅" if cid in row["gt_docs"] else "  "
            print(f"    {mark} {cid} — {_lookup_text(cid, client, cache)}")

        print(f"  [{combo2}] rank={row['rank_of_hit_2']}  top3:")
        for cid in row["ranked_ids_top5_2"][:3]:
            mark = "✅" if cid in row["gt_docs"] else "  "
            print(f"    {mark} {cid} — {_lookup_text(cid, client, cache)}")

    out_path = OUT_DIR / f"diff_{combo1}_vs_{combo2}.csv"
    changed.reset_index().to_csv(out_path, index=False)
    print(f"\n✅ diff 저장: {out_path}")

    return changed


def main():
    client = get_qdrant_client()
    model = load_embed_model()
    laws_ref = load_laws_ref()
    gold_standard = load_gold_standard()

    print(f"실버 스탠다드 {len(gold_standard)}개 로드 완료")

    reranker1, reranker2 = load_rerankers(device=RERANKER_DEVICE)
    cache = SweepCache()

    all_rows: list[dict] = []
    for combo in COMBOS:
        print(f"\n▶ {combo.name} 실행 중 ({combo.kind}, alpha={combo.alpha}, "
              f"r1={combo.use_reranker1}, r2={combo.use_reranker2}"
              + (f", jo_top_k={combo.jo_top_k})" if combo.kind == "cascaded" else ")"))
        rows = run_combo(combo, client, model, laws_ref, reranker1, reranker2, gold_standard, cache)
        all_rows.extend(rows)

    df = pd.DataFrame(all_rows)
    out_path = OUT_DIR / "query_level_results.csv"
    # ranked_ids_top5/gt_docs는 list라 CSV에는 문자열로 직렬화해서 저장 (분석용 df는 메모리에 그대로 유지)
    df_to_save = df.copy()
    df_to_save["gt_docs"] = df_to_save["gt_docs"].apply(json.dumps, ensure_ascii=False)
    df_to_save["ranked_ids_top5"] = df_to_save["ranked_ids_top5"].apply(json.dumps, ensure_ascii=False)
    df_to_save.to_csv(out_path, index=False)
    print(f"\n✅ 쿼리 단위 전체 결과 저장: {out_path}")

    for combo1, combo2 in DIFF_PAIRS:
        print_and_save_diff(df, combo1, combo2, client, cache)


if __name__ == "__main__":
    main()

Overwriting /content/yoonha_rag_query_diff.py


In [29]:
!python /content/yoonha_rag_query_diff.py

🗄️  로컬(파일 기반) Qdrant 사용: /content/qdrant_local
📦 임베딩 모델 로드: BAAI/bge-m3
Fetching 30 files: 100% 30/30 [00:00<00:00, 5648.89it/s]
Download complete: : 0.00B [00:00, ?B/s]
Loading weights: 100% 391/391 [00:08<00:00, 46.94it/s]
  ⚠️  laws_ref.json 없음: /data/hn_seed/law_refs.json
실버 스탠다드 50개 로드 완료
📦 Re-ranker 1단계 로드: Dongjin-kr/ko-reranker
Loading weights: 100% 393/393 [00:00<00:00, 1325.37it/s]
📦 Re-ranker 2단계 로드: BAAI/bge-reranker-v2-m3
Loading weights: 100% 393/393 [00:00<00:00, 823.03it/s]

▶ horag_r1_only 실행 중 (ho, alpha=0.6, r1=True, r2=False)

▶ horag_r1_r2 실행 중 (ho, alpha=0.6, r1=True, r2=True)

▶ cascaded_a1_k30 실행 중 (cascaded, alpha=1.0, r1=True, r2=False, jo_top_k=30)

▶ cascaded_a1_k40 실행 중 (cascaded, alpha=1.0, r1=True, r2=False, jo_top_k=40)

✅ 쿼리 단위 전체 결과 저장: /content/query_level_results.csv

  horag_r1_only  vs  horag_r1_r2
  총 50개 쿼리 중 25개 쿼리에서 순위 변화 발생

[o07] 🔻 악화  RR: 1.0 → 0.0
  질의: 회사 홈페이지에 게시된 개인정보 처리방침에 정보주체의 열람청구권 행사 방법에 관한 안내가 빠져 있습니다. 어떤 조항을 근거로 이 내용이 포함돼야
  정답: [

In [33]:
import pandas as pd

df = pd.read_csv("/content/query_level_results.csv")
summary = df.groupby("combo")[["rr", "hit@1", "hit@5", "hit@10"]].mean()
summary.columns = ["MRR", "Recall@1", "Recall@5", "Recall@10"]
print(summary.round(4))

                    MRR  Recall@1  Recall@5  Recall@10
combo                                                 
cascaded_a1_k30  0.7012      0.56      0.86       0.90
cascaded_a1_k40  0.7179      0.58      0.88       0.92
horag_r1_only    0.6985      0.56      0.88       0.92
horag_r1_r2      0.6547      0.52      0.88       0.94
